# SortAugment AI: Recyclable Material Image Augmenter

**Domain:** Waste Management  |  **Course:** Generative AI

**Problem.** A sorting classifier trained on a few clean photos of recyclables (TrashNet) breaks on a real sorting line, because real items are dim or glary (*lighting*), dirty or partly hidden (*contamination*), and crushed or rotated (*material condition*).

**Idea.** Create extra training images that contain these variations, then check whether a classifier trained on them does better on real-world data.

We build the extra images in two ways and compare them:
1. **Classical augmentation** (Albumentations + small OpenCV functions): fast, simple pixel tricks.
2. **Generative AI augmentation** (Stable Diffusion XL, img2img and inpainting): the model "repaints" the photo with dirt, wetness, crumpling, dim light and so on, while the object stays recognizable.

### Pipeline (the notebook follows these sections in order)
| # | Section | What happens |
|---|---|---|
| 1 | Setup | GPU check, Google Drive, one CONFIG cell, seeds |
| 2 | Data | TrashNet (train/val/test) and TACO (real-world test), class mapping |
| 3 | Baseline | ResNet18 on original TrashNet only: the "before" number |
| 4 | Classical augmentation | Lighting / contamination / material-condition transforms |
| 5 | Generative augmentation | SDXL img2img + inpainting with a prompt bank |
| 6 | Quality filter | CLIP + baseline classifier reject images that stopped looking like their class |
| 7 | Retrain 4 models | baseline, classical, generative, classical+generative |
| 8 | Evaluation | accuracy, macro-F1, confusion matrices, robustness test, optional Grad-CAM |
| 9 | Gradio demo | Upload an image, pick a condition, see augmentations and predictions |
| 10 | Export | results.csv, figures, best model, requirements, auto-summary |

### How to run this notebook
1. **First run with `DEBUG = True`** (5 images per class). This tests every cell in a few minutes. Then set `DEBUG = False` for the real run.
2. Run cells top to bottom. Every long step skips work that already exists in your Drive, so after a Colab disconnect you can just re-run from the top.
3. **Every number in this notebook is computed by the code when you run it.** Nothing is pre-filled.

## MANUAL STEP 0: Faculty rubric / extra deliverables

The prompt used to build this notebook had a placeholder for extra faculty requirements and it was **not filled in**, so none are assumed.

**You do:** paste the faculty rubric here (edit this cell), then tick off which section covers each item.

```
FACULTY RUBRIC (paste here):
- ...
```

Examples of things rubrics often ask for, and where this notebook covers them: dataset description (Section 2), baseline vs improved results (Sections 3, 7, 8), prompts used (Section 5 prompt bank + `gen_metadata.csv`), failure analysis (Section 6 rejected images, Section 8 confusion matrices), demo (Section 9), report numbers (Section 10 `summary.md`).

# 1. Setup, GPU check, Drive mount, config

**What this section does (in simple terms).**
- Installs the libraries we need. Colab already has PyTorch, so we do **not** reinstall it.
- Checks that a GPU is attached. Stable Diffusion XL on a CPU would take hours per image.
- Mounts Google Drive. Colab's local disk is wiped when a session ends, so *everything important is saved to Drive*.
- Puts every setting in **one CONFIG cell** so you change values in one place.
- Fixes random seeds so that re-running gives (nearly) the same result.

**MANUAL STEP:** In Colab choose **Runtime > Change runtime type > T4 GPU** before running anything.

**Restart note.** The install cell upgrades packages to their latest versions. If Colab then shows **"Restart session"**, click it and **continue from the next cell (the imports cell). Do not re-run the install cell.** Everything after it works without it.

In [ ]:
# ---- Install libraries (pure Python, so this cell has no notebook "magics") ----
import subprocess, sys

# torch / torchvision are already installed in Colab. We only add the rest.
PIP_PACKAGES = ("diffusers transformers accelerate safetensors albumentations gradio "
                "scikit-learn datasets huggingface_hub matplotlib pandas tqdm opencv-python-headless requests")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PIP_PACKAGES.split()], check=True)
print("\nInstall finished.")
print("If Colab shows 'Restart session', click it and continue from the NEXT cell (the imports cell).")
print("Do NOT re-run this install cell after the restart: it is not needed again in this session.")

In [ ]:
# ---- Imports ----
import os, sys, subprocess, json, math, random, time, gc, shutil, zipfile, warnings, zlib
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

def sh(cmd, check=True):
    """Run a shell command and stream its output live. (Defined here, not in the install cell, so it
    still exists after a Colab 'Restart session' when you skip the install cell.)"""
    print("$", cmd)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {cmd}")
    return proc.returncode

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

try:
    from google.colab import drive  # only exists inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

### The CONFIG cell
Everything you might want to change lives here.

- **`DEBUG`**: `True` = tiny run (5 images per class, 2 epochs, 2 source images per class for generation) to test the whole pipeline fast. Set to `False` for the real experiment.
- **Strength** (`GEN_STRENGTHS`): how far Stable Diffusion is allowed to move away from the original photo. `0.0` = identical, `1.0` = ignore the photo. We use about 0.3 to 0.6 so the object (and its class label) stays recognizable.
- **Guidance** (`GEN_GUIDANCE`): how strongly the model follows the text prompt. Higher means more literal but sometimes unnatural.
- **`GEN_SRC_PER_CLASS`**: how many original training images per class we send through Stable Diffusion. This is the main knob for run time (each source image produces one generated image per axis).

**MANUAL STEP:** choose values you are happy with. The defaults are sized for the free T4, but I have *not* been able to time them on a T4, so the notebook prints its own measured seconds-per-image and ETA during generation. If it is too slow, lower `GEN_SRC_PER_CLASS`.

In [ ]:
CONFIG = dict(
    # ---- run mode ----
    DEBUG=False,                      # <-- set to False for the real run
    DEBUG_PER_CLASS=5,               # images per class (TrashNet and TACO) in DEBUG mode
    SEED=42,

    # ---- data ----
    CLASS_NAMES=["cardboard", "glass", "metal", "paper", "plastic", "trash"],
    AXES=["lighting", "contamination", "condition"],   # the 3 variation axes from the problem statement
    IMG_SIZE=224,                    # classifier input size (square)
    VAL_FRAC=0.15, TEST_FRAC=0.15,   # TrashNet split: 70 / 15 / 15
    TACO_USE_640=True,               # download TACO's smaller 640px images (much faster); boxes are rescaled
    TACO_PAD=0.10,                   # extra context around each TACO box (fraction of box size)
    TACO_MIN_CROP=40,                # skip crops smaller than this (pixels, shorter side)
    TACO_MAX_PER_CLASS=None,         # e.g. 300 to cap per class; None = keep all

    # ---- paths ----
    DRIVE_ROOT="/content/drive/MyDrive/SortAugment_AI",
    LOCAL_ROOT="./SortAugment_AI_local",   # only used when NOT running in Colab

    # ---- classifier training (same for all 4 models) ----
    MODEL_NAME="resnet18",           # "resnet18" or "mobilenet_v2"
    EPOCHS=10, DEBUG_EPOCHS=2,
    BATCH_SIZE=64, LR=3e-4, WEIGHT_DECAY=1e-4, LABEL_SMOOTHING=0.1,
    NUM_WORKERS=0,                   # images are cached in RAM, so 0 is fine and avoids RAM duplication

    # ---- classical augmentation ----
    CLASSICAL_PER_IMAGE_PER_AXIS=1,  # augmented copies per source image per axis
    CLASSICAL_USE_ALL_TRAIN=False,   # False = same source images as the generative pool (fair comparison)
    MATCH_BUDGET=True,               # classical pool is trimmed to the same per-class/axis counts as the KEPT generative pool

    # ---- generative augmentation (Stable Diffusion) ----
    GEN_MODEL_ID="stabilityai/stable-diffusion-xl-base-1.0",
    GEN_VAE_ID="madebyollin/sdxl-vae-fp16-fix",           # avoids black/NaN images with fp16 SDXL
    GEN_FALLBACK_IDS=["stable-diffusion-v1-5/stable-diffusion-v1-5", "sd-legacy/stable-diffusion-v1-5"],
    GEN_FORCE_SD15=False,            # True = skip SDXL and use SD 1.5 directly (for testing the fallback)
    GEN_SIZE_SDXL=(768, 576),        # (width, height), multiples of 64, same 4:3 shape as TrashNet
    GEN_SIZE_SD15=(512, 384),
    GEN_STEPS=30,                    # real denoising steps = steps x strength
    GEN_BATCH_SDXL=2, GEN_BATCH_SD15=4,
    GEN_SRC_PER_CLASS=90, DEBUG_GEN_SRC_PER_CLASS=2,
    GEN_STRENGTHS={"lighting": [0.30, 0.38, 0.45],        # img2img strength choices per axis
                   "contamination": [0.35, 0.45, 0.55],
                   "condition": [0.40, 0.50, 0.60]},
    GEN_INPAINT_STRENGTHS=[0.45, 0.52, 0.60],             # inpainting strength choices
    GEN_INPAINT_FRACTION={"lighting": 0.0, "contamination": 0.5, "condition": 0.4},  # share of images made by inpainting
    GEN_GUIDANCE=[5.0, 7.0],

    # ---- quality filter (Section 6) ----
    CLIP_MODEL_ID="openai/clip-vit-base-patch32",
    USE_CLIP_FILTER=True, USE_BASE_FILTER=True,
    CLIP_MIN_PROB=0.20,              # keep if CLIP gives the true class >= this prob, or CLIP's top-1 is the true class
    BASE_MIN_PROB=0.20,              # baseline = SANITY CHECK only (chance is 1/6 = 0.17). A high value would keep only images the
                                     # baseline already recognises (= easy ones) and defeat the purpose. Set USE_BASE_FILTER=False to let CLIP decide alone.

    # ---- evaluation ----
    ROBUST_SEVERITIES=[1, 2, 3],
    BOOTSTRAP_N=1000,
    RUN_GRADCAM=True,
    BEST_BY="robust_mean",           # how the "best model" is picked: "robust_mean", "taco_f1" or "trashnet_f1"
)

cfg = SimpleNamespace(**CONFIG)
if cfg.DEBUG:
    # In DEBUG the baseline sees ~18 training images for 2 epochs: too weak to judge anything, it would reject
    # almost every generated image and Section 7 would then skip models C and D.
    cfg.USE_BASE_FILTER = False
    CONFIG["USE_BASE_FILTER"] = False      # keep the exported config.json truthful
cfg.RUN_NAME = "debug" if cfg.DEBUG else "full"
cfg.EPOCHS_EFF = cfg.DEBUG_EPOCHS if cfg.DEBUG else cfg.EPOCHS
cfg.GEN_SRC_EFF = cfg.DEBUG_GEN_SRC_PER_CLASS if cfg.DEBUG else cfg.GEN_SRC_PER_CLASS
CLASS_NAMES, AXES, N_CLASSES = cfg.CLASS_NAMES, cfg.AXES, len(cfg.CLASS_NAMES)
print(f"Run mode: {cfg.RUN_NAME.upper()} | epochs: {cfg.EPOCHS_EFF} | generation sources per class: {cfg.GEN_SRC_EFF}")

In [ ]:
# ---- Drive mount and folders ----
if IN_COLAB:
    drive.mount("/content/drive")   # MANUAL STEP: click "Connect to Google Drive" and allow access
    ROOT = Path(cfg.DRIVE_ROOT)
else:
    ROOT = Path(cfg.LOCAL_ROOT).resolve()   # local testing outside Colab

DATA_DIR = ROOT / "data"                      # downloaded datasets (shared by DEBUG and full runs)
RUN_DIR = ROOT / "runs" / cfg.RUN_NAME        # everything produced by this run mode
CLASSICAL_DIR = RUN_DIR / "classical_aug"
GEN_DIR = RUN_DIR / "generated"
MODEL_DIR = RUN_DIR / "models"
FIG_DIR = RUN_DIR / "figures"
TABLE_DIR = RUN_DIR / "tables"
EXPORT_DIR = RUN_DIR / "export"
TACO_CROP_DIR = RUN_DIR / "taco_crops"
for d in [DATA_DIR, RUN_DIR, CLASSICAL_DIR, GEN_DIR, MODEL_DIR, FIG_DIR, TABLE_DIR, EXPORT_DIR, TACO_CROP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Everything is saved under:", ROOT)

In [ ]:
# ---- GPU check ----
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB | torch {torch.__version__}")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: no GPU found. In Colab use Runtime > Change runtime type > T4 GPU, then re-run.")
    print("         CPU is OK for DEBUG classifier steps but Stable Diffusion (Section 5) will be far too slow.")
USE_AMP = DEVICE.type == "cuda"   # mixed precision speeds up training on the T4

In [ ]:
# ---- Small helpers used everywhere ----
def seed_everything(seed):
    """Fix all random number generators so runs are repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

def stable_seed(*parts):
    """Turn any tuple of values into the same 32-bit seed every time (unlike Python's hash())."""
    return zlib.crc32("|".join(str(p) for p in parts).encode()) & 0x7FFFFFFF

def free_gpu():
    """Release cached GPU memory (call after deleting big models)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("saved figure:", path)

seed_everything(cfg.SEED)
print("Seeds fixed to", cfg.SEED)

# 2. Data loading, class mapping, splits, sample visualization

**Two datasets, two jobs.**

| Dataset | Used for | Why |
|---|---|---|
| **TrashNet** (2,527 photos, 6 classes, items on a white poster board) | train / val / test | clean, simple, easy to learn from |
| **TACO** (litter photographed outdoors, boxes + 60 fine categories) | a separate **messy real-world test set** | different lighting, backgrounds, dirt, crushed items; the classifier never trains on it |

**The golden rule (no data leakage).** We split TrashNet into train / val / test **first**. Only the **train** part is ever augmented or sent to Stable Diffusion. Val, test and TACO stay untouched. If augmented copies of a test image leaked into training, the test score would be fake.

**How TACO becomes a 6-class test set.** TACO has bounding boxes and 60 categories (e.g. "Drink can", "Glass bottle"). We crop every annotated object and map its category onto our 6 classes with a dictionary you can read and edit.

### How the downloads work (verified, not guessed)
- **TrashNet:** the Hugging Face dataset repo `garythung/trashnet` contains `dataset-resized.zip` (about 43 MB, 512x384 photos). The notebook downloads that file with `huggingface_hub` and unzips it into Drive.
- **TACO:** the official repo is `github.com/pedropro/TACO`. We `git clone` it to get `data/annotations.json` (COCO format). The official `download.py` fetches images from Flickr but **stops with an error on the first dead link**, so this notebook uses its own tolerant, resumable downloader that reads the same `flickr_url` / `flickr_640_url` fields from the annotations. Images are saved at `data/<file_name>` exactly like the official script.

## MANUAL STEP 2a: only if an automatic download fails
- **TrashNet:** open `https://huggingface.co/datasets/garythung/trashnet`, download `dataset-resized.zip`, unzip it, and put the six class folders (`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) in **`MyDrive/SortAugment_AI/data/trashnet/`**.
- **TACO:** the TACO README also links a full copy on Zenodo (DOI `10.5281/zenodo.3587843`). Download and unzip it so that you have **`MyDrive/SortAugment_AI/data/TACO/data/annotations.json`** and the `batch_*` image folders next to it.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder):
    """All image files under a folder (ignores macOS junk like __MACOSX and ._files)."""
    out = []
    for p in Path(folder).rglob("*"):
        if p.suffix.lower() in IMG_EXT and "__MACOSX" not in p.parts and not p.name.startswith("._"):
            out.append(p)
    return sorted(out)

def find_class_root(base, class_names):
    """Find the folder that directly contains one sub-folder per class (case-insensitive)."""
    base = Path(base)
    if not base.exists():
        return None
    candidates = [base] + sorted(p for p in base.rglob("*") if p.is_dir() and "__MACOSX" not in p.parts)
    for c in candidates:
        subs = {d.name.lower() for d in c.iterdir() if d.is_dir()}
        if all(n in subs for n in class_names):
            return c
    return None

In [ ]:
# ---- TrashNet download (resumable: skipped if the folder already exists) ----
TRASHNET_DIR = DATA_DIR / "trashnet"

def download_trashnet():
    if find_class_root(TRASHNET_DIR, CLASS_NAMES) is not None:
        print("TrashNet already present:", find_class_root(TRASHNET_DIR, CLASS_NAMES))
        return
    from huggingface_hub import hf_hub_download
    raw_dir = DATA_DIR / "trashnet_raw"
    raw_dir.mkdir(exist_ok=True)
    print("Downloading dataset-resized.zip from Hugging Face (garythung/trashnet) ...")
    zip_path = hf_hub_download(repo_id="garythung/trashnet", filename="dataset-resized.zip",
                               repo_type="dataset", local_dir=str(raw_dir))
    TRASHNET_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(TRASHNET_DIR)
    print("Unzipped to", TRASHNET_DIR)

try:
    download_trashnet()
except Exception as e:
    print("AUTOMATIC TRASHNET DOWNLOAD FAILED:", repr(e))
    print("-> Do MANUAL STEP 2a (TrashNet) above, then re-run this cell.")

TRASHNET_ROOT = find_class_root(TRASHNET_DIR, CLASS_NAMES)
assert TRASHNET_ROOT is not None, "TrashNet class folders not found. Do MANUAL STEP 2a."
print("TrashNet class folders found in:", TRASHNET_ROOT)

In [ ]:
# ---- TACO: get annotations.json (git clone is the official route) ----
TACO_REPO = DATA_DIR / "TACO"
TACO_ANN = TACO_REPO / "data" / "annotations.json"

if not TACO_ANN.exists():
    try:
        if TACO_REPO.exists() and not any(TACO_REPO.iterdir()):
            TACO_REPO.rmdir()
        if not TACO_REPO.exists():
            sh(f"git clone --depth 1 https://github.com/pedropro/TACO.git {TACO_REPO}")
    except Exception as e:
        print("git clone failed:", repr(e))
assert TACO_ANN.exists(), "TACO annotations.json not found. Do MANUAL STEP 2a (TACO)."

with open(TACO_ANN) as f:
    TACO = json.load(f)
print(f"TACO annotations: {len(TACO['images'])} images, {len(TACO['annotations'])} boxes, {len(TACO['categories'])} categories")

### TACO category to 6-class mapping
This is the dictionary that decides what each TACO object counts as. A TACO test label is only useful if a person would agree on it, so we use **two lists**:

- `TACO_TO_6`: categories with a clear home in our 6 classes.
- `TACO_EXCLUDED`: categories we **drop on purpose**, each with a one-line reason. Two kinds: (1) items with no matching class (battery, food waste, shoe, rope, cigarette, TACO's "unlabeled litter"), which would turn "trash" into a grab-bag; (2) **multi-material or contaminated items whose label is arguable** (drink carton / Tetra Pak, used tissues, plastic-coated paper bag, aluminium blister pack).

The "trash" class is therefore small and packaging-like (foam cups and containers, styrofoam, carded blister packs). That is a deliberate trade-off: a cleaner label for fewer images.

**MANUAL STEP 2b:** read both lists below. **Your faculty may ask about the borderline ones, so be ready to defend (or change) each decision.** Move a category between the lists if you disagree. The cell after them prints the excluded categories, any TACO category that is in neither list (also dropped), and names that do not exist in the JSON (a spelling mismatch to fix). Later the notebook warns you if a class ends up with fewer than 30 crops.

In [ ]:
# Keys are lower-case TACO category names. Values are our 6 classes.
TACO_TO_6 = {
    # ---- cardboard ----
    "toilet tube": "cardboard", "other carton": "cardboard", "egg carton": "cardboard",
    "corrugated carton": "cardboard", "meal carton": "cardboard",
    "pizza box": "cardboard",
    # ---- glass ----
    "glass bottle": "glass", "broken glass": "glass", "glass cup": "glass", "glass jar": "glass",
    # ---- metal ----
    "aluminium foil": "metal", "metal bottle cap": "metal",
    "food can": "metal", "aerosol": "metal", "drink can": "metal", "metal lid": "metal",
    "pop tab": "metal", "scrap metal": "metal",
    # ---- paper ----
    "paper cup": "paper", "magazine paper": "paper", "wrapping paper": "paper",
    "normal paper": "paper", "paper bag": "paper", "paper straw": "paper",
    # ---- plastic ----
    "other plastic bottle": "plastic", "clear plastic bottle": "plastic", "plastic bottle cap": "plastic",
    "disposable plastic cup": "plastic", "other plastic cup": "plastic", "plastic lid": "plastic",
    "other plastic": "plastic", "plastic film": "plastic", "six pack rings": "plastic",
    "garbage bag": "plastic", "other plastic wrapper": "plastic", "single-use carrier bag": "plastic",
    "polypropylene bag": "plastic", "crisp packet": "plastic", "spread tub": "plastic",
    "tupperware": "plastic", "disposable food container": "plastic", "other plastic container": "plastic",
    "plastic glooves": "plastic", "plastic gloves": "plastic",   # TACO spells it "glooves"
    "plastic utensils": "plastic", "squeezable tube": "plastic", "plastic straw": "plastic",
    # ---- trash: only packaging-like non-recyclables that look alike (see TACO_EXCLUDED for what is left out) ----
    "carded blister pack": "trash", "foam cup": "trash", "foam food container": "trash", "styrofoam piece": "trash",
}

# Categories we deliberately do NOT put in any of the 6 classes. Their boxes are dropped from the TACO test set.
# Reason: a test label is only meaningful if a person would agree on it. These either have no class in our 6
# (so "trash" would become a grab-bag of unrelated things) or are multi-material / contaminated items whose label is arguable.
TACO_EXCLUDED = {
    "battery": "hazardous item, not a TrashNet-style 'trash' object",
    "food waste": "organic, not a recyclable-sorting category",
    "shoe": "textile/rubber item, no matching class",
    "rope & strings": "textile/plastic mix, no matching class",
    "cigarette": "tiny, no matching class",
    "unlabeled litter": "TACO's own 'unknown' bucket: the true material is unknown",
    "drink carton": "multi-layer (paperboard + plastic + foil, e.g. Tetra Pak): cardboard, paper and plastic are all arguable",
    "tissues": "used tissues are contaminated, so not clean 'paper' in the recycling sense",
    "plastified paper bag": "paper with a plastic coating: paper or plastic is arguable",
    "aluminium blister pack": "aluminium + plastic laminate: metal or plastic is arguable",
}
for _k in TACO_EXCLUDED:
    TACO_TO_6.pop(_k, None)            # also removes the name if someone re-adds it to the dict above by mistake
assert set(TACO_TO_6.values()) <= set(CLASS_NAMES)

taco_cat_names = {c["id"]: c["name"].strip() for c in TACO["categories"]}
json_names_lower = {n.lower() for n in taco_cat_names.values()}
excluded_lower = {k.lower() for k in TACO_EXCLUDED}
unmapped = sorted(n for n in taco_cat_names.values() if n.lower() not in TACO_TO_6 and n.lower() not in excluded_lower)
missing_in_json = sorted(k for k in list(TACO_TO_6) + list(TACO_EXCLUDED) if k not in json_names_lower)
print(f"Mapped categories: {sum(n.lower() in TACO_TO_6 for n in taco_cat_names.values())} / {len(taco_cat_names)}")
print("Deliberately excluded (see TACO_EXCLUDED for reasons):", sorted(n for n in taco_cat_names.values() if n.lower() in excluded_lower))
print("TACO categories with NO mapping and NOT in the exclusion list (their boxes will be dropped too):", unmapped or "none")
print("Names in TACO_TO_6 / TACO_EXCLUDED not found in this TACO json (harmless if they are just alternative spellings):", missing_in_json or "none")

In [ ]:
# ---- Build TACO annotation table, choose the boxes we will use ----
def build_taco_annotation_table():
    img_info = {im["id"]: im for im in TACO["images"]}
    rows = []
    for a in TACO["annotations"]:
        cname = taco_cat_names.get(a["category_id"], "")
        cls = TACO_TO_6.get(cname.lower())
        if cls is None:
            continue
        im = img_info[a["image_id"]]
        rows.append(dict(ann_id=a["id"], image_id=a["image_id"], file_name=im["file_name"],
                         ann_w=im.get("width"), ann_h=im.get("height"),
                         bbox=a["bbox"], taco_category=cname, label=cls, label_id=CLASS_NAMES.index(cls),
                         flickr_url=im.get("flickr_url"), flickr_640_url=im.get("flickr_640_url")))
    return pd.DataFrame(rows)

taco_ann = build_taco_annotation_table()

# choose which boxes to use (deterministic)
def choose_taco_boxes(df):
    rng = np.random.default_rng(cfg.SEED)
    parts = []
    for cls in CLASS_NAMES:
        sub = df[df.label == cls]
        limit = cfg.DEBUG_PER_CLASS if cfg.DEBUG else cfg.TACO_MAX_PER_CLASS
        if limit is not None and len(sub) > limit:
            sub = sub.iloc[rng.permutation(len(sub))[:limit]]
        parts.append(sub)
    return pd.concat(parts).reset_index(drop=True)

taco_sel = choose_taco_boxes(taco_ann)
print("TACO boxes available per class:\n", taco_ann.label.value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict())
print("TACO boxes selected per class: \n", taco_sel.label.value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict())
print("Images to download:", taco_sel.file_name.nunique())

In [ ]:
# ---- Tolerant, resumable TACO image downloader ----
import requests
from concurrent.futures import ThreadPoolExecutor

def _fetch(url, dest):
    """Download one URL to dest. Returns True on success. Skips files that already exist."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return True
    if not url:
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200 or len(r.content) < 1000:
            return False
        tmp = dest.with_suffix(dest.suffix + ".part")
        tmp.write_bytes(r.content)
        os.replace(tmp, dest)          # a half-written file is never mistaken for a finished one
        return True
    except Exception:
        return False

def download_taco_images(df, workers=8):
    """Download every image referenced in df. Tries the 640px URL first (if enabled), then the original URL."""
    imgs = df.drop_duplicates("file_name")[["file_name", "flickr_url", "flickr_640_url"]]
    def job(row):
        dest = TACO_REPO / "data" / row.file_name
        urls = [row.flickr_640_url, row.flickr_url] if cfg.TACO_USE_640 else [row.flickr_url, row.flickr_640_url]
        return row.file_name, any(_fetch(u, dest) for u in urls)
    with ThreadPoolExecutor(workers) as ex:
        results = list(tqdm(ex.map(job, imgs.itertuples()), total=len(imgs), desc="TACO images"))
    failed = [n for n, ok in results if not ok]
    print(f"Downloaded/available: {len(results) - len(failed)} / {len(results)}. Failed (dead links?): {len(failed)}")
    if len(failed) > 0.3 * len(results):
        print("MANUAL STEP 2a: many TACO links failed. Use the Zenodo copy described above, then re-run this cell.")
    return failed

failed_taco = download_taco_images(taco_sel)

In [ ]:
# ---- Crop the annotated objects into a 6-class test set (resumable) ----
def crop_taco(df):
    rows = []
    for r in tqdm(df.itertuples(), total=len(df), desc="Cropping TACO objects"):
        src = TACO_REPO / "data" / r.file_name
        out = TACO_CROP_DIR / r.label / f"{r.image_id}_{r.ann_id}.jpg"
        if out.exists():
            rows.append(dict(path=str(out), label=r.label, label_id=r.label_id, taco_category=r.taco_category))
            continue
        if not src.exists():
            continue                                              # image failed to download
        try:
            im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        except Exception:
            continue
        sx = im.width / r.ann_w if r.ann_w else 1.0               # boxes are in original-image pixels; we may hold the 640px copy
        sy = im.height / r.ann_h if r.ann_h else 1.0
        x, y, w, h = r.bbox
        x, y, w, h = x * sx, y * sy, w * sx, h * sy
        padx, pady = w * cfg.TACO_PAD, h * cfg.TACO_PAD
        box = (max(0, int(x - padx)), max(0, int(y - pady)), min(im.width, int(x + w + padx)), min(im.height, int(y + h + pady)))
        if min(box[2] - box[0], box[3] - box[1]) < cfg.TACO_MIN_CROP:
            continue                                              # too small to recognise
        out.parent.mkdir(parents=True, exist_ok=True)
        im.crop(box).save(out, quality=95)
        rows.append(dict(path=str(out), label=r.label, label_id=r.label_id, taco_category=r.taco_category))
    return pd.DataFrame(rows)

taco_test = crop_taco(taco_sel)
taco_test.to_csv(TABLE_DIR / "taco_test.csv", index=False)
print("TACO test crops:", len(taco_test))
taco_counts = taco_test.label.value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
print(taco_counts.to_dict())
small_classes = [c for c, n in taco_counts.items() if n < 30]
if small_classes and not cfg.DEBUG:
    print("WARNING: fewer than 30 TACO crops for", small_classes, "-> macro-F1 on TACO is noisy for these classes.",
          "Consider moving some categories back from TACO_EXCLUDED into TACO_TO_6.")

In [ ]:
# ---- TrashNet dataframe and train/val/test split ----
def build_trashnet_df(root):
    rows = []
    folders = {d.name.lower(): d for d in Path(root).iterdir() if d.is_dir()}
    for ci, cname in enumerate(CLASS_NAMES):
        for p in list_images(folders[cname]):
            rows.append(dict(path=str(p), label=cname, label_id=ci))
    return pd.DataFrame(rows)

def split_per_class(df, val_frac, test_frac, seed):
    """Stratified split done class by class (works even with only 5 images per class)."""
    rng = np.random.default_rng(seed)
    parts = []
    for cname in CLASS_NAMES:
        sub = df[df.label == cname].iloc[rng.permutation((df.label == cname).sum())]
        n = len(sub)
        n_val = max(1, int(round(val_frac * n)))
        n_test = max(1, int(round(test_frac * n)))
        sub = sub.copy()
        sub["split"] = ["val"] * n_val + ["test"] * n_test + ["train"] * (n - n_val - n_test)
        parts.append(sub)
    return pd.concat(parts).reset_index(drop=True)

SPLIT_CSV = TABLE_DIR / "trashnet_splits.csv"
if SPLIT_CSV.exists():
    trashnet_all = pd.read_csv(SPLIT_CSV)
    print("Loaded existing split from", SPLIT_CSV)
else:
    tn = build_trashnet_df(TRASHNET_ROOT)
    if cfg.DEBUG:
        rng = np.random.default_rng(cfg.SEED)
        tn = pd.concat([g.iloc[rng.permutation(len(g))[:cfg.DEBUG_PER_CLASS]] for _, g in tn.groupby("label")])
    trashnet_all = split_per_class(tn, cfg.VAL_FRAC, cfg.TEST_FRAC, cfg.SEED)
    trashnet_all.to_csv(SPLIT_CSV, index=False)
    print("Saved split to", SPLIT_CSV)

train_df = trashnet_all[trashnet_all.split == "train"].reset_index(drop=True)
val_df = trashnet_all[trashnet_all.split == "val"].reset_index(drop=True)
test_df = trashnet_all[trashnet_all.split == "test"].reset_index(drop=True)

# safety checks: no image may appear in two splits
assert not (set(train_df.path) & set(val_df.path)) and not (set(train_df.path) & set(test_df.path)) and not (set(val_df.path) & set(test_df.path))
print("Split sizes -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df), "| TACO test:", len(taco_test))
counts = pd.DataFrame({"train": train_df.label.value_counts(), "val": val_df.label.value_counts(),
                       "test": test_df.label.value_counts(), "taco_test": taco_test.label.value_counts()}).reindex(CLASS_NAMES).fillna(0).astype(int)
display(counts)

In [ ]:
# ---- Sample visualization: TrashNet vs TACO ----
def show_samples(df, title, n=4, fname=None):
    fig, axes = plt.subplots(N_CLASSES, n, figsize=(2.2 * n, 2.0 * N_CLASSES))
    for ci, cname in enumerate(CLASS_NAMES):
        sub = df[df.label == cname].head(n)
        for j in range(n):
            ax = axes[ci, j]
            ax.axis("off")
            if j < len(sub):
                ax.imshow(ImageOps.exif_transpose(Image.open(sub.iloc[j].path)).convert("RGB"))
            if j == 0:
                ax.set_title(cname, loc="left", fontsize=10)
    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    if fname:
        save_fig(fig, fname)
    plt.show()

show_samples(train_df, "TrashNet train samples (clean, white background)", fname="01_trashnet_samples.png")
show_samples(taco_test, "TACO test crops (messy real world)", fname="02_taco_samples.png")

# 3. Baseline classifier (the "before" result)

**Goal.** Train a classifier on the **original TrashNet training images only** and measure how well it does on (a) the clean TrashNet test set and (b) the messy TACO test set. Every later experiment is compared against this.

**What is happening, in simple terms.**
- We use **ResNet18 pretrained on ImageNet**. It already knows edges, textures and shapes, so we only need to *fine-tune* it on our 6 classes. This is called transfer learning.
- Images are resized to 224x224. Our only "augmentation" for **all four models** is a random left-right flip, so the comparison between models is fair.
- To keep Colab fast we decode each image **once**, resize it, and keep it in RAM as a NumPy array (a cache). Reading thousands of small files from Drive repeatedly would be very slow.
- We train for a fixed number of epochs and keep the checkpoint with the best **validation macro-F1**. Validation is the clean TrashNet val split, which is never augmented.
- Training is **resumable**: a checkpoint is saved every epoch in Drive. If a finished model already exists, it is loaded instead of retrained.

**Accuracy vs macro-F1.** Accuracy is the share of correct predictions. *Macro-F1* averages the F1 score of each class equally, so a classifier cannot look good by only being right on the biggest class. TACO is very unbalanced (lots of plastic), so macro-F1 matters.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

IMG_CACHE = {}   # (path, size) -> uint8 array of shape (size, size, 3)

def load_img_array(path, size=None):
    """Load an image, fix its EXIF rotation, resize to size x size (squash), return uint8 RGB array."""
    size = size or cfg.IMG_SIZE
    key = (str(path), size)
    if key not in IMG_CACHE:
        im = ImageOps.exif_transpose(Image.open(path)).convert("RGB").resize((size, size), Image.BILINEAR)
        IMG_CACHE[key] = np.asarray(im, dtype=np.uint8)
    return IMG_CACHE[key]

def load_df_images(df, desc="Loading images"):
    """Return an array (N, H, W, 3) for every row of df, using a thread pool because Drive reads are slow."""
    paths = list(df.path)
    with ThreadPoolExecutor(8) as ex:
        arrays = list(tqdm(ex.map(load_img_array, paths), total=len(paths), desc=desc, leave=False))
    return np.stack(arrays) if arrays else np.zeros((0, cfg.IMG_SIZE, cfg.IMG_SIZE, 3), np.uint8)

def to_tensor_batch(x_uint8):
    """uint8 (B,H,W,3) numpy -> normalised float tensor (B,3,H,W) on DEVICE."""
    x = torch.from_numpy(x_uint8).to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
    return (x - IMAGENET_MEAN.to(DEVICE)) / IMAGENET_STD.to(DEVICE)

class ArrayDataset(torch.utils.data.Dataset):
    """Wraps cached image arrays. Training adds a random horizontal flip (same for all 4 models)."""
    def __init__(self, X, y, train=False):
        self.X, self.y, self.train = X, np.asarray(y), train
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i]).permute(2, 0, 1)
        if self.train and random.random() < 0.5:
            x = torch.flip(x, dims=[2])
        return x, int(self.y[i])

In [ ]:
def make_model(name=None, n_classes=N_CLASSES, pretrained=True):
    """Pretrained ImageNet backbone with a new 6-way output layer."""
    from torchvision import models
    name = name or cfg.MODEL_NAME
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    elif name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    else:
        raise ValueError(name)
    return m.to(DEVICE)

@torch.no_grad()
def predict_proba(model, X, bs=128):
    """Class probabilities for an array of uint8 images (N,H,W,3)."""
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = to_tensor_batch(X[i:i + bs])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(xb)
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, N_CLASSES), np.float32)

def compute_metrics(y_true, y_pred):
    """Accuracy, macro-F1, per-class report and confusion matrix."""
    labels = list(range(N_CLASSES))
    return dict(
        acc=float(accuracy_score(y_true, y_pred)),
        macro_f1=float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        report=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
        cm=confusion_matrix(y_true, y_pred, labels=labels),
    )

In [ ]:
def train_model(tag, train_frame, val_frame, epochs=None, resume=True):
    """
    Fine-tune a classifier and save it to Drive.  Returns (model, history_dataframe).
    - Resumable: a finished model is just loaded; an interrupted run continues from its last epoch.
    - Model selection uses macro-F1 on the clean val set only.
    """
    epochs = epochs or cfg.EPOCHS_EFF
    best_path, last_path = MODEL_DIR / f"{tag}_best.pt", MODEL_DIR / f"{tag}_last.pt"
    hist_path = TABLE_DIR / f"history_{tag}.csv"

    if resume and best_path.exists() and hist_path.exists():
        print(f"[{tag}] finished model found in Drive -> loading instead of retraining")
        model = make_model()
        model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["state_dict"])
        return model, pd.read_csv(hist_path)

    seed_everything(cfg.SEED)
    Xtr, ytr = load_df_images(train_frame, f"[{tag}] train images"), train_frame.label_id.values
    Xva, yva = load_df_images(val_frame, f"[{tag}] val images"), val_frame.label_id.values
    drop_last = len(Xtr) % cfg.BATCH_SIZE == 1          # BatchNorm cannot train on a batch of one
    g = torch.Generator().manual_seed(cfg.SEED)
    loader = torch.utils.data.DataLoader(ArrayDataset(Xtr, ytr, train=True), batch_size=cfg.BATCH_SIZE, shuffle=True,
                                         num_workers=cfg.NUM_WORKERS, generator=g, drop_last=drop_last)

    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
    start_epoch, best_f1, history = 0, -1.0, []

    if resume and last_path.exists():
        ck = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["state_dict"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start_epoch, best_f1, history = ck["epoch"] + 1, ck["best_f1"], ck["history"]
        print(f"[{tag}] resuming from epoch {start_epoch}")

    for epoch in range(start_epoch, epochs):
        model.train()
        run_loss, n_seen, t0 = 0.0, 0, time.time()
        for xb, yb in tqdm(loader, desc=f"[{tag}] epoch {epoch + 1}/{epochs}", leave=False):
            xb = (xb.to(DEVICE).float() / 255.0 - IMAGENET_MEAN.to(DEVICE)) / IMAGENET_STD.to(DEVICE)
            yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            run_loss += loss.item() * len(yb); n_seen += len(yb)
        sched.step()

        pv = predict_proba(model, Xva).argmax(1)
        m = compute_metrics(yva, pv)
        history.append(dict(epoch=epoch + 1, train_loss=run_loss / max(n_seen, 1), val_acc=m["acc"], val_macro_f1=m["macro_f1"], seconds=time.time() - t0))
        print(f"[{tag}] epoch {epoch + 1}/{epochs}  loss {history[-1]['train_loss']:.4f}  val acc {m['acc']:.3f}  val macro-F1 {m['macro_f1']:.3f}")

        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]
            torch.save(dict(state_dict=model.state_dict(), tag=tag, arch=cfg.MODEL_NAME, class_names=CLASS_NAMES,
                            img_size=cfg.IMG_SIZE, epoch=epoch + 1, val_macro_f1=best_f1), best_path)
        torch.save(dict(state_dict=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        epoch=epoch, best_f1=best_f1, history=history), last_path)

    hist = pd.DataFrame(history)
    hist.to_csv(hist_path, index=False)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["state_dict"])   # back to the best epoch
    if last_path.exists():
        last_path.unlink()                                                                                 # finished: drop the resume file
    return model, hist

In [ ]:
# ---- Train the baseline (original TrashNet train only) ----
MODELS, N_TRAIN, HISTORY = {}, {}, {}      # filled by every training run

def get_model(tag):
    """Return a trained model from memory, or load its best checkpoint from Drive (useful after a Colab restart)."""
    if tag not in MODELS:
        ck = torch.load(MODEL_DIR / f"{tag}_best.pt", map_location=DEVICE, weights_only=False)
        m = make_model(); m.load_state_dict(ck["state_dict"]); m.eval()
        MODELS[tag] = m
    return MODELS[tag]

MODELS["A_baseline"], HISTORY["A_baseline"] = train_model("A_baseline", train_df, val_df)
N_TRAIN["A_baseline"] = len(train_df)

In [ ]:
# ---- The "before" result: accuracy and macro-F1 on TrashNet test and TACO test ----
X_tn_test = load_df_images(test_df, "TrashNet test"); y_tn_test = test_df.label_id.values
X_taco = load_df_images(taco_test, "TACO test");      y_taco = taco_test.label_id.values

def quick_eval(tag):
    out = {}
    for name, X, y in [("trashnet_test", X_tn_test, y_tn_test), ("taco_test", X_taco, y_taco)]:
        out[name] = compute_metrics(y, predict_proba(get_model(tag), X).argmax(1))
    return out

base_eval = quick_eval("A_baseline")
before = pd.DataFrame({
    "set": ["TrashNet test (clean)", "TACO test (messy real world)"],
    "n_images": [len(y_tn_test), len(y_taco)],
    "accuracy": [base_eval["trashnet_test"]["acc"], base_eval["taco_test"]["acc"]],
    "macro_F1": [base_eval["trashnet_test"]["macro_f1"], base_eval["taco_test"]["macro_f1"]],
}).round(4)
display(before)
before.to_csv(TABLE_DIR / "baseline_before.csv", index=False)
print("Expect a big gap between the clean and the messy test set: that gap is exactly the problem this project tries to shrink.")

# 4. Classical augmentation (Albumentations), organised by the three variation axes

**What is classical augmentation?** Ordinary image processing: change brightness, add noise, rotate. It is fast, free and needs no GPU. We organise the transforms by the three problems named in the project statement:

| Axis | What happens on a real sorting line | Transforms we use |
|---|---|---|
| **Lighting** | dim warehouse, glare, hard shadows, warm/cold lamps | brightness/contrast, shadow, colour temperature, low light, glare |
| **Contamination** | dirt, wet smears, items partly hidden | noise, blur, random occlusion patches, stain-like overlays |
| **Material condition** | crushed, bent, tilted, only partly in view | rotation, perspective, elastic / crumple-like distortion, partial crop |

**How it is built.**
- Each axis is a dictionary of small functions `f(image, rng) -> image`. Where Albumentations has a good transform we use it; where it does not (glare, stains, crumple, colour temperature) we wrote a short NumPy/OpenCV function.
- Each Albumentations transform is wrapped so that, if a future Albumentations version rejects an argument, a simple NumPy version takes over and the notebook prints a message instead of crashing.
- For one augmented copy we pick **1 or 2 random transforms from one axis**, so the image really represents *that* axis.
- Augmented copies are written to Drive (resumable). **Only train images are ever augmented.**

**Known limitation (please read).** These tricks only change pixels. A real crushed can has new folds, highlights and shadows that a pixel filter cannot invent. That is the gap generative AI (Section 5) is meant to fill.

In [ ]:
import cv2
import albumentations as A

AUG_RNG_SEED = cfg.SEED

# ---------- low-level helpers ----------
def smooth_field(h, w, scale, rng):
    """Low-frequency random field in [0,1]: random noise on a coarse grid, blown up smoothly."""
    gh, gw = max(2, h // scale), max(2, w // scale)
    f = cv2.resize(rng.random((gh, gw)).astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
    f = (f - f.min()) / (f.max() - f.min() + 1e-6)
    return f

def clip_u8(x):
    return np.clip(x, 0, 255).astype(np.uint8)

# ---------- NumPy versions (also used as fallbacks for Albumentations ones) ----------
def np_brightness_contrast(img, rng):
    a = rng.uniform(0.6, 1.4)                      # contrast
    b = rng.uniform(-50, 50)                       # brightness
    return clip_u8((img.astype(np.float32) - 128) * a + 128 + b)

def np_shadow(img, rng):
    h, w = img.shape[:2]
    pts = np.array([[rng.uniform(0, w), 0], [rng.uniform(0, w), 0], [rng.uniform(0, w), h], [rng.uniform(0, w), h]], np.int32)
    mask = np.zeros((h, w), np.float32)
    cv2.fillPoly(mask, [pts], 1.0)
    mask = cv2.GaussianBlur(mask, (0, 0), max(h, w) * 0.03)
    return clip_u8(img.astype(np.float32) * (1 - rng.uniform(0.3, 0.6) * mask[..., None]))

def np_noise(img, rng):
    sigma = rng.uniform(8, 35)
    return clip_u8(img.astype(np.float32) + rng.normal(0, sigma, img.shape))

def np_blur(img, rng):
    if rng.random() < 0.5:                         # Gaussian blur = out of focus
        k = int(rng.choice([3, 5, 7, 9]))
        return cv2.GaussianBlur(img, (k, k), 0)
    L, ang = int(rng.integers(5, 15)), rng.uniform(0, 180)   # motion blur = belt moving fast
    ker = np.zeros((L, L), np.float32); ker[L // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((L / 2 - 0.5, L / 2 - 0.5), ang, 1.0)
    ker = cv2.warpAffine(ker, M, (L, L)); ker /= ker.sum() + 1e-6
    return cv2.filter2D(img, -1, ker)

def np_rotate(img, rng):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-45, 45), 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

def np_perspective(img, rng):
    h, w = img.shape[:2]
    s = rng.uniform(0.05, 0.15) * min(h, w)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = src + rng.uniform(-s, s, src.shape).astype(np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

def np_elastic(img, rng, amp=None):
    """Smooth random warp of the pixel grid (like a deformed, soft object)."""
    h, w = img.shape[:2]
    amp = amp if amp is not None else rng.uniform(0.02, 0.05) * min(h, w)
    dx = (smooth_field(h, w, 24, rng) - 0.5) * 2 * amp
    dy = (smooth_field(h, w, 24, rng) - 0.5) * 2 * amp
    gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    return cv2.remap(img, gx + dx, gy + dy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

# ---------- custom transforms that Albumentations does not offer ----------
def color_temperature(img, rng):
    """Warm (orange) or cold (blue) lamp: scale red and blue channels in opposite directions."""
    k = rng.uniform(-0.28, 0.28)
    out = img.astype(np.float32)
    out[..., 0] *= 1 + k
    out[..., 2] *= 1 - k
    return clip_u8(out)

def low_light(img, rng):
    """Dim scene: gamma curve + overall darkening + a little sensor noise."""
    x = (img.astype(np.float32) / 255.0) ** rng.uniform(1.6, 2.8) * 255.0 * rng.uniform(0.45, 0.8)
    return clip_u8(x + rng.normal(0, rng.uniform(2, 6), img.shape))

def glare(img, rng):
    """A bright specular hot-spot, like a lamp reflecting off a shiny surface."""
    h, w = img.shape[:2]
    cx, cy = rng.uniform(0.2, 0.8) * w, rng.uniform(0.2, 0.8) * h
    rx, ry = rng.uniform(0.08, 0.25) * w, rng.uniform(0.08, 0.25) * h
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    spot = np.exp(-(((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2))
    return clip_u8(img.astype(np.float32) + 255.0 * spot[..., None] * rng.uniform(0.6, 1.0))

def occlusion_patches(img, rng):
    """Random patches that hide part of the item (other waste, a hand, a label...)."""
    h, w = img.shape[:2]
    out = img.copy()
    for _ in range(int(rng.integers(1, 5))):
        pw, ph = int(rng.uniform(0.08, 0.28) * w), int(rng.uniform(0.08, 0.28) * h)
        x0, y0 = int(rng.uniform(0, w - pw)), int(rng.uniform(0, h - ph))
        color = tuple(int(c) for c in rng.integers(20, 200, 3))
        if rng.random() < 0.5:
            cv2.rectangle(out, (x0, y0), (x0 + pw, y0 + ph), color, -1)
        else:
            cv2.ellipse(out, (x0 + pw // 2, y0 + ph // 2), (pw // 2, ph // 2), float(rng.uniform(0, 180)), 0, 360, color, -1)
    return out

STAIN_COLORS = [(90, 60, 30), (30, 30, 30), (170, 150, 60), (80, 50, 30), (110, 100, 90)]   # mud, oil, grease, coffee, dust

def stain_overlay(img, rng):
    """Soft irregular dirty blobs blended over the image."""
    h, w = img.shape[:2]
    f = smooth_field(h, w, 48, rng)
    thr = rng.uniform(0.62, 0.82)
    mask = np.clip((f - thr) / 0.10, 0, 1)                       # blobs where the field is high
    mask = cv2.GaussianBlur(mask, (0, 0), 2.0)[..., None]
    color = np.array(STAIN_COLORS[int(rng.integers(len(STAIN_COLORS)))], np.float32)
    alpha = rng.uniform(0.4, 0.75)
    return clip_u8(img.astype(np.float32) * (1 - alpha * mask) + color * alpha * mask)

def crumple(img, rng):
    """Crumple-like look: a strong smooth warp plus soft crease shading (light/dark ridges from a noise field)."""
    h, w = img.shape[:2]
    warped = np_elastic(img, rng, amp=rng.uniform(0.04, 0.08) * min(h, w))
    field = cv2.GaussianBlur(smooth_field(h, w, 12, rng), (0, 0), 3)
    gy, gx = np.gradient(field)
    shade = 0.7 * gx + 0.7 * gy                                   # pretend the light comes from the top-left
    shade = np.clip(shade / (np.percentile(np.abs(shade), 99) + 1e-6), -1, 1)
    return clip_u8(warped.astype(np.float32) * (1 + rng.uniform(0.15, 0.35) * shade[..., None]))

def partial_crop(img, rng):
    """Only part of the item is in view: crop 55-85% of the image and scale it back up."""
    h, w = img.shape[:2]
    frac = rng.uniform(0.55, 0.85)
    ch, cw = int(h * frac), int(w * frac)
    y0, x0 = int(rng.uniform(0, h - ch)), int(rng.uniform(0, w - cw))
    return cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_LINEAR)

In [ ]:
def alb_op(makers, fallback, name):
    """
    Wrap an Albumentations transform as f(img, rng) -> img.
    `makers` is one factory or a list of factories (different argument styles for different Albumentations versions).
    The first one that builds AND runs is used; if none works, the NumPy fallback is used and a message is printed.
    """
    makers = makers if isinstance(makers, (list, tuple)) else [makers]
    for make in makers:
        try:
            t = make()
            t(image=np.zeros((48, 48, 3), np.uint8))          # smoke test
            def f(img, rng, t=t):
                return t(image=img)["image"]
            return f
        except Exception:
            continue
    print(f"[albumentations] '{name}' not usable in this version; using NumPy fallback.")
    return fallback

AXIS_OPS = {
    "lighting": {
        "brightness_contrast": alb_op(lambda: A.RandomBrightnessContrast(brightness_limit=(-0.4, 0.4), contrast_limit=(-0.4, 0.4), p=1.0), np_brightness_contrast, "RandomBrightnessContrast"),
        "shadow": alb_op(lambda: A.RandomShadow(p=1.0), np_shadow, "RandomShadow"),
        "color_temperature": color_temperature,
        "low_light": low_light,
        "glare": glare,
    },
    "contamination": {
        "noise": alb_op([lambda: A.GaussNoise(std_range=(0.03, 0.12), p=1.0),          # Albumentations 2.x
                         lambda: A.GaussNoise(var_limit=(10.0, 60.0), p=1.0)],       # Albumentations 1.x
                        np_noise, "GaussNoise"),
        "blur": alb_op(lambda: A.OneOf([A.GaussianBlur(blur_limit=(3, 9), p=1.0), A.MotionBlur(blur_limit=(5, 13), p=1.0)], p=1.0), np_blur, "GaussianBlur/MotionBlur"),
        "occlusion_patches": occlusion_patches,
        "stain_overlay": stain_overlay,
    },
    "condition": {
        "rotation": alb_op(lambda: A.Rotate(limit=(-45, 45), border_mode=cv2.BORDER_REFLECT_101, p=1.0), np_rotate, "Rotate"),
        "perspective": np_perspective,                       # own OpenCV version: identical across Albumentations versions
        "elastic": alb_op([lambda: A.ElasticTransform(alpha=120, sigma=10, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
                           lambda: A.ElasticTransform(alpha=120, sigma=10, p=1.0)], np_elastic, "ElasticTransform"),
        "crumple": crumple,
        "partial_crop": partial_crop,
    },
}
for ax, ops in AXIS_OPS.items():
    print(f"{ax:14s}: {', '.join(ops)}")

In [ ]:
def choose_ops(axis, rng):
    """Pick 1 or 2 different transform names from one axis."""
    names = list(AXIS_OPS[axis])
    k = 1 if rng.random() < 0.5 else 2
    return [names[i] for i in rng.permutation(len(names))[:k]]

def apply_classical(img, axis, rng, ops=None):
    """Apply the chosen transforms of one axis to a uint8 RGB image. Returns (new_image, op_names)."""
    ops = ops or choose_ops(axis, rng)
    out = img
    for name in ops:
        out = AXIS_OPS[axis][name](out, rng)
    return np.ascontiguousarray(out), ops

# ---- quick visual check on a few training images ----
def show_classical_examples(df, n_src=3, n_aug=4, fname="03_classical_examples.png"):
    fig, axes = plt.subplots(n_src * len(AXES), n_aug + 1, figsize=(2.3 * (n_aug + 1), 2.1 * n_src * len(AXES)))
    rng = np.random.default_rng(cfg.SEED)
    row = 0
    for si in range(n_src):
        src = np.asarray(ImageOps.exif_transpose(Image.open(df.iloc[si * max(1, len(df) // n_src)].path)).convert("RGB"))
        for axis in AXES:
            axes[row, 0].imshow(src); axes[row, 0].set_title(f"original", fontsize=8)
            for j in range(n_aug):
                aug, ops = apply_classical(src, axis, rng)
                axes[row, j + 1].imshow(aug); axes[row, j + 1].set_title(f"{axis}: {'+'.join(ops)}", fontsize=7)
            for a in axes[row]:
                a.axis("off")
            row += 1
    plt.tight_layout(); save_fig(fig, fname); plt.show()

show_classical_examples(train_df)

### Choose the source images (shared by classical and generative pools)
Stable Diffusion is slow, so we do not send every training image through it. For each class we pick up to `GEN_SRC_PER_CLASS` training images (random but fixed by the seed). **The classical pool is built from the same images**, so the two methods are compared on equal footing. (Set `CLASSICAL_USE_ALL_TRAIN=True` in CONFIG if you also want classical copies of the whole training set.)

These are always **train** images, never val or test.

In [ ]:
def pick_source_images(df, per_class, seed):
    rng = np.random.default_rng(seed)
    parts = []
    for cname in CLASS_NAMES:
        sub = df[df.label == cname]
        parts.append(sub.iloc[rng.permutation(len(sub))[:per_class]])
    return pd.concat(parts).reset_index(drop=True)

gen_sources = pick_source_images(train_df, cfg.GEN_SRC_EFF, cfg.SEED)
gen_sources.to_csv(TABLE_DIR / "generation_sources.csv", index=False)
print("Source images per class:", gen_sources.label.value_counts().reindex(CLASS_NAMES).to_dict())
assert set(gen_sources.path) <= set(train_df.path), "sources must come from the TRAIN split only"

In [ ]:
def build_classical_pool(src_df, k_per_axis):
    """Write augmented copies to Drive and return a metadata dataframe. Existing files are skipped (resumable)."""
    rows = []
    for r in tqdm(src_df.itertuples(), total=len(src_df), desc="Classical augmentation"):
        img = None
        for axis in AXES:
            for k in range(k_per_axis):
                stem = Path(r.path).stem
                out = CLASSICAL_DIR / axis / r.label / f"{stem}__{axis}_{k}.jpg"
                rng = np.random.default_rng(stable_seed(cfg.SEED, "classical", r.path, axis, k))
                ops = choose_ops(axis, rng)                   # same draws every time -> metadata is reproducible
                if not out.exists():
                    if img is None:
                        img = np.asarray(ImageOps.exif_transpose(Image.open(r.path)).convert("RGB"))
                    aug, _ = apply_classical(img, axis, rng, ops=ops)
                    out.parent.mkdir(parents=True, exist_ok=True)
                    Image.fromarray(aug).save(out, quality=95)
                rows.append(dict(path=str(out), label=r.label, label_id=r.label_id, axis=axis, ops="+".join(ops), source=r.path, origin="classical"))
    return pd.DataFrame(rows)

classical_src = train_df if cfg.CLASSICAL_USE_ALL_TRAIN else gen_sources
classical_pool = build_classical_pool(classical_src, cfg.CLASSICAL_PER_IMAGE_PER_AXIS)
classical_pool.to_csv(TABLE_DIR / "classical_pool.csv", index=False)
print("Classical pool size:", len(classical_pool))
print(classical_pool.groupby(["axis"]).size().to_dict())

# 5. Generative AI augmentation with Stable Diffusion XL (img2img + inpainting)

**The idea, in simple terms.** Stable Diffusion creates images by starting from noise and removing it step by step, guided by a text prompt. Two ways to use it for augmentation:

- **img2img**: start from a *real photo with a bit of noise added* instead of pure noise. The **strength** setting says how much noise: low strength (0.3 to 0.6) keeps the object's shape and colour but lets the model repaint the lighting, add dirt, or make it look crushed. We want exactly this, so the **class label stays valid**.
- **inpainting**: we draw a random blob mask over part of the photo, and the model repaints *only that region* (for example a grease stain, or a plastic film covering part of the item).

**Prompt bank.** For every class and every axis we build many prompts from small pieces (item description + axis phrase + conveyor-belt scene). Axis phrases: *dim warehouse light, glare, wet, dirty/stained/greasy, crushed/crumpled, partially covered, conveyor-belt background*.

**Safety nets built in.**
- **Resumable:** every image is saved the moment it is made, and `gen_metadata.csv` is appended after each batch. Re-running the cell skips finished images.
- **Memory:** SDXL runs in fp16 with a fixed fp16-safe VAE, VAE slicing, and small batches. If the GPU runs out of memory the batch is split; if even a single image fails, the notebook switches to **Stable Diffusion 1.5** automatically (smaller, 512x384).
- **Reproducible:** each image has its own recorded seed.

**Honest limits.** Diffusion models can still change the object more than you want (a glass bottle can turn into a jar, a stain can look like a different material). That is exactly why Section 6 filters the results. Also, Stable Diffusion does not know the TrashNet class names, it only follows the prompt, so prompt wording matters a lot.

## MANUAL STEP 5a: before the first run
- Make sure the runtime has the **T4 GPU**.
- The first run downloads SDXL (roughly 7 GB). This repeats in every new Colab session because Colab's local disk is wiped, so expect several minutes of download.
- Optional: log in to Hugging Face for faster, more reliable downloads (SDXL base is not gated, so this is not required). Run `from huggingface_hub import login; login()` in a new cell.

In [ ]:
# ---------------- Prompt bank ----------------
CLASS_DESC = {   # kept generic on purpose: TrashNet items vary a lot inside each class
    "cardboard": ["a cardboard item", "a recyclable cardboard piece"],
    "glass": ["a glass item", "a recyclable glass container"],
    "metal": ["a metal item", "a recyclable metal object"],
    "paper": ["a paper item", "a recyclable paper piece"],
    "plastic": ["a plastic item", "a recyclable plastic object"],
    "trash": ["a piece of rubbish", "a non-recyclable waste item"],
}

AXIS_PHRASES = {   # img2img: describe the whole scene with the variation we want
    "lighting": [
        "in dim warehouse light, low light, deep shadows",
        "under harsh overhead fluorescent light with strong glare and reflections",
        "under warm yellow industrial lamps with uneven shadows",
        "under cold blue-tinted light in a recycling plant",
    ],
    "contamination": [
        "dirty, stained and covered in grease and food residue",
        "wet and muddy with dirt smudges",
        "dusty and grimy with oily stains",
        "with sticky residue and coffee stains",
    ],
    "condition": [
        "crushed and crumpled",
        "bent, flattened and worn out",
        "torn and deformed, partially covered by other waste",
        "partially covered by a thin plastic bag",
    ],
}

INPAINT_PHRASES = {   # inpainting: describe only what should appear inside the masked patch
    "contamination": ["dirty grease stain and smudges", "wet mud and food residue", "dark oily stain", "dried sticky residue"],
    "condition": ["partly covered by a thin transparent plastic film", "partly covered by dust and debris",
                  "partly covered by a dirty cloth", "partly hidden under crumpled waste"],
}

SCENES = ["on a dark rubber conveyor belt in a recycling facility",
          "on a metal sorting table in a waste sorting plant",
          "on a grey conveyor belt, industrial setting"]
QUALITY = "realistic photo, sharp focus"
NEGATIVE_PROMPT = "cartoon, illustration, painting, drawing, text, watermark, logo, people, hands, different object, extra objects"

def build_prompt_bank():
    bank = {c: {a: [] for a in AXES} for c in CLASS_NAMES}
    for c in CLASS_NAMES:
        for a in AXES:
            i = 0
            for desc in CLASS_DESC[c]:
                for ph in AXIS_PHRASES[a]:
                    bank[c][a].append(f"a photo of {desc} {ph}, {SCENES[i % len(SCENES)]}, {QUALITY}")
                    i += 1
    return bank

def build_inpaint_bank():
    bank = {c: {a: [] for a in AXES} for c in CLASS_NAMES}
    for c in CLASS_NAMES:
        for a, phrases in INPAINT_PHRASES.items():
            for desc in CLASS_DESC[c]:
                for ph in phrases:
                    bank[c][a].append(f"a photo of {desc}, {ph}, {QUALITY}")
    return bank

PROMPT_BANK, INPAINT_BANK = build_prompt_bank(), build_inpaint_bank()
with open(GEN_DIR / "prompt_bank.json", "w") as f:
    json.dump(dict(img2img=PROMPT_BANK, inpaint=INPAINT_BANK, negative=NEGATIVE_PROMPT), f, indent=1)

n_i2i = sum(len(v) for c in PROMPT_BANK.values() for v in c.values())
n_inp = sum(len(v) for c in INPAINT_BANK.values() for v in c.values())
print(f"Prompt bank: {n_i2i} img2img prompts + {n_inp} inpainting prompts (saved to {GEN_DIR / 'prompt_bank.json'})")
for axis in AXES:
    print(f"\n[{axis}] example for 'plastic':\n  ", PROMPT_BANK["plastic"][axis][0])
print("\nNegative prompt:", NEGATIVE_PROMPT)

## MANUAL STEP 5b: review the prompt bank
Read the prompts above. This is the part of the project where **your judgement matters most**:
- Do the phrases describe conditions you would really see on a sorting line?
- Could a prompt push the model to change the *material* (e.g. "wrapped in a plastic bag" on a glass item)? If so, soften or remove it.
- You can edit `CLASS_DESC`, `AXIS_PHRASES`, `INPAINT_PHRASES`, `SCENES` in the cell above and re-run it. Generated files from an earlier prompt set stay in Drive, so **delete the `runs/<mode>/generated` folder in Drive** if you want a clean regeneration after big prompt changes.

In [ ]:
# ---------------- Masks and the generation plan ----------------
def make_mask(w, h, seed):
    """Random soft blob mask (white = repaint). Covers roughly 5-20% of the image."""
    rng = np.random.default_rng(seed)
    m = np.zeros((h, w), np.uint8)
    for _ in range(int(rng.integers(1, 3))):
        cx, cy = int(rng.uniform(0.25, 0.75) * w), int(rng.uniform(0.25, 0.75) * h)
        ax, ay = int(rng.uniform(0.08, 0.20) * w), int(rng.uniform(0.08, 0.20) * h)
        cv2.ellipse(m, (cx, cy), (ax, ay), float(rng.uniform(0, 180)), 0, 360, 255, -1)
    return Image.fromarray(cv2.GaussianBlur(m, (0, 0), 6))

META_COLS = ["out_path", "src_path", "label", "label_id", "axis", "mode", "prompt", "strength", "guidance", "steps", "seed", "model", "gen_w", "gen_h"]
GEN_META_CSV = GEN_DIR / "gen_metadata.csv"

def plan_generation_tasks(src_df):
    """One task per (source image, axis). Everything random is derived from a stable seed, so the plan is the same on every run."""
    tasks = []
    for r in src_df.itertuples():
        for axis in AXES:
            rng = np.random.default_rng(stable_seed(cfg.SEED, "gen", r.path, axis))
            inpaint_possible = len(INPAINT_BANK[r.label][axis]) > 0
            mode = "inpaint" if (inpaint_possible and rng.random() < cfg.GEN_INPAINT_FRACTION.get(axis, 0.0)) else "img2img"
            if mode == "img2img":
                strength = float(cfg.GEN_STRENGTHS[axis][int(rng.integers(len(cfg.GEN_STRENGTHS[axis])))])
                bank = PROMPT_BANK[r.label][axis]
            else:
                strength = float(cfg.GEN_INPAINT_STRENGTHS[int(rng.integers(len(cfg.GEN_INPAINT_STRENGTHS)))])
                bank = INPAINT_BANK[r.label][axis]
            prompt = bank[int(rng.integers(len(bank)))]
            guidance = float(cfg.GEN_GUIDANCE[int(rng.integers(len(cfg.GEN_GUIDANCE)))])
            seed = int(rng.integers(0, 2**31 - 1))
            out = GEN_DIR / axis / r.label / f"{Path(r.path).stem}__{axis}_{mode}.jpg"
            tasks.append(dict(out_path=str(out), src_path=r.path, label=r.label, label_id=r.label_id, axis=axis, mode=mode,
                              prompt=prompt, strength=strength, guidance=guidance, steps=cfg.GEN_STEPS, seed=seed))
    return pd.DataFrame(tasks)

def load_done_meta():
    """Rows of finished images (the CSV row exists AND the image file exists)."""
    if not GEN_META_CSV.exists():
        return pd.DataFrame(columns=META_COLS)
    m = pd.read_csv(GEN_META_CSV).drop_duplicates("out_path", keep="last")
    return m[m.out_path.map(lambda p: Path(p).exists())]

gen_plan = plan_generation_tasks(gen_sources)
done_meta = load_done_meta()
todo = gen_plan[~gen_plan.out_path.isin(done_meta.out_path)]
print(f"Generation plan: {len(gen_plan)} images in total | already done: {len(gen_plan) - len(todo)} | still to do: {len(todo)}")
print(gen_plan.groupby(["axis", "mode"]).size().unstack(fill_value=0))
print("\nThe progress bar during generation shows measured seconds per image and an ETA, so you will know the real cost after the first batch.")
print("If it is too slow for your free-tier time, lower GEN_SRC_PER_CLASS in CONFIG (each source image makes 3 generated images: one per axis).")

In [ ]:
# ---------------- Load Stable Diffusion (SDXL, with SD 1.5 fallback) ----------------
from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting, AutoencoderKL

GEN = {}   # holds the loaded pipelines: name, i2i, inp, size, batch

def release_generators():
    GEN.clear()
    free_gpu()

def _load_sdxl():
    kw = dict(torch_dtype=torch.float16, use_safetensors=True)
    try:   # the stock SDXL VAE can produce NaN / black images in fp16; this community-fixed VAE does not
        kw["vae"] = AutoencoderKL.from_pretrained(cfg.GEN_VAE_ID, torch_dtype=torch.float16)
    except Exception as e:
        print("fp16-fix VAE could not be loaded, using the default SDXL VAE:", repr(e)[:120])
    i2i = AutoPipelineForImage2Image.from_pretrained(cfg.GEN_MODEL_ID, variant="fp16", **kw).to("cuda")
    inp = AutoPipelineForInpainting.from_pipe(i2i)          # shares the same weights: no extra GPU memory
    return i2i, inp

def _load_sd15():
    last = None
    for mid in cfg.GEN_FALLBACK_IDS:                        # the old runwayml repo was removed from the Hub; these are the maintained copies
        try:
            i2i = AutoPipelineForImage2Image.from_pretrained(mid, torch_dtype=torch.float16, safety_checker=None,
                                                             requires_safety_checker=False).to("cuda")
            return i2i, AutoPipelineForInpainting.from_pipe(i2i)
        except Exception as e:
            print(f"could not load {mid}: {repr(e)[:150]}")
            last = e
    raise RuntimeError("No Stable Diffusion 1.5 fallback could be loaded") from last

def load_generators(force_sd15=False):
    """Load SDXL; on a GPU out-of-memory error fall back to SD 1.5."""
    release_generators()
    if not torch.cuda.is_available():
        raise RuntimeError("Stable Diffusion needs a GPU. Use Runtime > Change runtime type > T4 GPU.")
    loaded = None
    if not (force_sd15 or cfg.GEN_FORCE_SD15):
        try:
            loaded = ("sdxl", *_load_sdxl(), cfg.GEN_SIZE_SDXL, cfg.GEN_BATCH_SDXL)
        except torch.cuda.OutOfMemoryError:
            print("SDXL ran out of GPU memory -> falling back to Stable Diffusion 1.5")
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            print("SDXL ran out of memory -> falling back to Stable Diffusion 1.5")
        if loaded is None:
            release_generators()
    if loaded is None:
        loaded = ("sd15", *_load_sd15(), cfg.GEN_SIZE_SD15, cfg.GEN_BATCH_SD15)
    name, i2i, inp, size, batch = loaded
    try:
        i2i.vae.enable_slicing()      # decode the VAE one image at a time: saves memory
        i2i.vae.enable_tiling()
    except Exception:
        pass
    i2i.set_progress_bar_config(disable=True); inp.set_progress_bar_config(disable=True)
    GEN.update(name=name, i2i=i2i, inp=inp, size=size, batch=batch)
    print(f"Generator ready: {name} | image size {size} | batch {batch}")

In [ ]:
# ---------------- Core generation functions ----------------
def _run_batch(batch):
    """batch = list of task dicts that share the same mode / strength / guidance."""
    W, H = GEN["size"]
    mode, strength, guidance = batch[0]["mode"], batch[0]["strength"], batch[0]["guidance"]
    imgs = [ImageOps.exif_transpose(Image.open(t["src_path"])).convert("RGB").resize((W, H), Image.LANCZOS) for t in batch]
    gens = [torch.Generator(device="cuda").manual_seed(int(t["seed"])) for t in batch]
    kw = dict(prompt=[t["prompt"] for t in batch], negative_prompt=[NEGATIVE_PROMPT] * len(batch), image=imgs,
              strength=strength, guidance_scale=guidance, num_inference_steps=cfg.GEN_STEPS, generator=gens)
    if mode == "inpaint":
        kw["mask_image"] = [make_mask(W, H, t["seed"]) for t in batch]
        return GEN["inp"](**kw).images
    return GEN["i2i"](**kw).images

def safe_generate(batch):
    """Run a batch. On out-of-memory: split the batch; at batch size 1 switch from SDXL to SD 1.5."""
    try:
        return _run_batch(batch)
    except torch.cuda.OutOfMemoryError:
        free_gpu()
        if len(batch) > 1:
            mid = len(batch) // 2
            print(f"GPU out of memory with batch {len(batch)} -> splitting into {mid} + {len(batch) - mid}")
            return safe_generate(batch[:mid]) + safe_generate(batch[mid:])
        if GEN["name"] == "sdxl":
            print("GPU out of memory even for one image -> switching to Stable Diffusion 1.5")
            load_generators(force_sd15=True)
            return safe_generate(batch)
        raise

def run_generation(plan_df):
    done = load_done_meta()
    remaining = plan_df[~plan_df.out_path.isin(done.out_path)]
    if remaining.empty:
        print("Nothing to generate: all images already exist.")
        return
    if not GEN:
        load_generators()
    batches = []
    for _, g in remaining.groupby(["mode", "strength", "guidance"], sort=True):   # a batch must share these values
        recs = g.to_dict("records")
        batches += [recs[i:i + GEN["batch"]] for i in range(0, len(recs), GEN["batch"])]
    t_start, n_done = time.time(), 0
    pbar = tqdm(batches, desc="Generating")
    for batch in pbar:
        outs = safe_generate(batch)
        rows = []
        for t, im in zip(batch, outs):
            Path(t["out_path"]).parent.mkdir(parents=True, exist_ok=True)
            im.save(t["out_path"], quality=95)
            rows.append({**t, "model": GEN["name"], "gen_w": im.width, "gen_h": im.height})
        pd.DataFrame(rows)[META_COLS].to_csv(GEN_META_CSV, mode="a", header=not GEN_META_CSV.exists(), index=False)
        n_done += len(batch)
        el = time.time() - t_start
        pbar.set_postfix(sec_per_img=f"{el / n_done:.1f}", eta_min=f"{(len(remaining) - n_done) * el / n_done / 60:.0f}")

In [ ]:
# ---------------- RUN: generate the images ----------------
run_generation(gen_plan)

# tidy the metadata file: one row per existing image
gen_meta = load_done_meta().reset_index(drop=True)
gen_meta.to_csv(GEN_META_CSV, index=False)
gen_meta.to_csv(TABLE_DIR / "gen_metadata.csv", index=False)
print(f"Generated images on disk: {len(gen_meta)}")
display(gen_meta.groupby(["model", "axis", "mode"]).size().rename("images").to_frame())
release_generators()      # free the GPU before we train classifiers again

In [ ]:
# ---------------- Look at the results ----------------
def show_generated(meta, n=8, fname="04_generated_examples.png"):
    sample = meta.sample(min(n, len(meta)), random_state=cfg.SEED)
    fig, axes = plt.subplots(2, len(sample), figsize=(2.6 * len(sample), 5.4), squeeze=False)
    for j, r in enumerate(sample.itertuples()):
        axes[0, j].imshow(ImageOps.exif_transpose(Image.open(r.src_path)).convert("RGB"))
        axes[0, j].set_title(f"original ({r.label})", fontsize=8)
        axes[1, j].imshow(Image.open(r.out_path))
        axes[1, j].set_title(f"{r.axis}\n{r.mode} s={r.strength:.2f}", fontsize=8)
        axes[0, j].axis("off"); axes[1, j].axis("off")
    plt.tight_layout(); save_fig(fig, fname); plt.show()

show_generated(gen_meta)

## MANUAL STEP 5c: look at the generated images yourself
Before moving on, scroll through a few dozen files in `MyDrive/SortAugment_AI/runs/<mode>/generated/` (or just rerun the cell above for different samples by changing `n`). Ask:
- Is the object still the same kind of object?
- Does the picture show the *intended* condition (dim, dirty, crumpled)?
- Are there artefacts (melted shapes, text, weird colours, black images)?

If many images look wrong, adjust the prompts (5b) or lower the strengths in CONFIG and regenerate. The automatic quality filter in Section 6 is a safety net, not a replacement for looking.

# 6. Quality filter: reject generated images that no longer look like their class

**Why.** Generated images keep the *original label*. If Stable Diffusion turned a glass bottle into something unrecognisable, training on it with the label "glass" **teaches the classifier a lie**. So we check every generated image with two independent judges and only keep the ones that pass:

1. **CLIP zero-shot.** CLIP is a model trained on image-text pairs. We ask it "which of these 6 descriptions fits this picture best?" without any training on our data. An image passes if CLIP's top choice is the true class **or** CLIP gives the true class at least `CLIP_MIN_PROB`.
2. **The baseline classifier** (trained on original TrashNet only), used only as a **sanity check**. An image passes if the baseline gives the true class at least `BASE_MIN_PROB`. The default is **0.20**, only a little above chance (1/6 = 0.17), so it removes just the images the baseline sees as clearly something else.

An image is **kept only if it passes every enabled judge**. (You can switch judges on or off in CONFIG.)

**Things to understand about this filter.**
- **Why the baseline threshold is low.** The baseline can only recognise what it has already learned. If we kept only images it is confident about (say 50% or more), we would keep the *easy* images and throw away exactly the dim, dirty, crushed ones that augmentation is meant to add. That would bias the generative data toward what the model already knows. So the baseline is a weak safety net, and **CLIP does the real filtering**. If you prefer, set `USE_BASE_FILTER=False` and let CLIP decide alone.
- **DEBUG mode switches the baseline judge off automatically.** In DEBUG the baseline is trained on about 18 images for 2 epochs, so it is too weak to judge anything and would reject nearly everything, which would make Section 7 skip models C and D.
- CLIP is not perfect on waste items, especially the vague class "trash". That is why the cell below first measures CLIP's zero-shot accuracy on *real clean validation images*, so you can see how much to trust it.
- **Trash and glass are exempt from the CLIP judge.** In a previous run CLIP's zero-shot accuracy on real validation images was very low for these two classes (4.8% for trash, 41% for glass), so it would reject most of their generated images for no good reason. They are judged by the baseline sanity check only. That is a weaker check, and **in DEBUG mode (baseline judge off) these two classes are not filtered at all**. Always look at the kept and rejected trash and glass examples shown below.
- Both judges can **share the blind spots** we are trying to fix: either may reject a perfectly valid "dim light" image *because* it is dim. This can make the filter **too strict on exactly the hard variations**. Look at the keep rates per axis in the table below to see if this is happening.
- The thresholds are heuristics. **Tune them using the validation set or by eye, never by looking at test results.**

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_TEXTS = [
    "a photo of a piece of cardboard",
    "a photo of a glass bottle or glass jar",
    "a photo of a metal can or scrap metal",
    "a photo of a piece of paper",
    "a photo of a plastic bottle or plastic container",
    "a photo of rubbish or general waste",
]   # same order as CLASS_NAMES

clip_model = CLIPModel.from_pretrained(cfg.CLIP_MODEL_ID).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(cfg.CLIP_MODEL_ID)

@torch.no_grad()
def clip_probs(paths, bs=32):
    """Zero-shot class probabilities (N, 6) from CLIP for a list of image paths."""
    out = []
    for i in tqdm(range(0, len(paths), bs), desc="CLIP", leave=False):
        ims = [ImageOps.exif_transpose(Image.open(p)).convert("RGB") for p in paths[i:i + bs]]
        inp = clip_proc(text=CLIP_TEXTS, images=ims, return_tensors="pt", padding=True).to(DEVICE)
        out.append(clip_model(**inp).logits_per_image.softmax(dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, N_CLASSES))

# how trustworthy is CLIP on REAL clean images? (validation set, never test)
p_val = clip_probs(list(val_df.path))
clip_val_acc = pd.DataFrame({"label": val_df.label, "correct": p_val.argmax(1) == val_df.label_id.values}).groupby("label").correct.mean().reindex(CLASS_NAMES)
print(f"CLIP zero-shot accuracy on real TrashNet VAL images: {np.mean(p_val.argmax(1) == val_df.label_id.values):.3f}")
print("per class:", clip_val_acc.round(3).to_dict())

In [ ]:
# ---------------- Score every generated image ----------------
gen_meta = load_done_meta().reset_index(drop=True)
assert len(gen_meta) > 0, "No generated images found. Run Section 5 first."
gm = gen_meta.copy()
gm["path"] = gm.out_path

Xg = load_df_images(gm, "generated images")
pb = predict_proba(get_model("A_baseline"), Xg)
idx = np.arange(len(gm))
gm["base_p_label"] = pb[idx, gm.label_id.values]
gm["base_top1"] = pb.argmax(1)

pc = clip_probs(list(gm.out_path))
gm["clip_p_label"] = pc[idx, gm.label_id.values]
gm["clip_top1"] = pc.argmax(1)

base_ok = (gm.base_p_label >= cfg.BASE_MIN_PROB) if cfg.USE_BASE_FILTER else pd.Series(True, index=gm.index)
clip_ok = ((gm.clip_p_label >= cfg.CLIP_MIN_PROB) | (gm.clip_top1 == gm.label_id)) if cfg.USE_CLIP_FILTER else pd.Series(True, index=gm.index)
clip_ok = clip_ok | gm.label.isin(["trash", "glass"])   # CLIP was unreliable on these (4.8% / 41% on real val images)
gm["base_ok"], gm["clip_ok"] = base_ok, clip_ok
gm["keep"] = base_ok & clip_ok

gm.to_csv(TABLE_DIR / "gen_quality_scores.csv", index=False)
gen_kept = gm[gm.keep].reset_index(drop=True)
gen_kept = gen_kept.assign(origin="generative")
gen_kept.to_csv(TABLE_DIR / "gen_kept.csv", index=False)

print(f"Generated: {len(gm)} | kept: {len(gen_kept)} ({100 * gm.keep.mean():.1f}%) | rejected: {len(gm) - len(gen_kept)}")
print(f"Rejected by baseline judge: {(~gm.base_ok).sum()} | rejected by CLIP judge: {(~gm.clip_ok).sum()}")

In [ ]:
# ---------------- Keep / reject rates ----------------
rates_axis_class = gm.pivot_table(index="label", columns="axis", values="keep", aggfunc="mean").reindex(CLASS_NAMES)[AXES] * 100
rates_overall = gm.groupby("axis").keep.mean().reindex(AXES) * 100
rates_mode = gm.groupby(["axis", "mode"]).keep.agg(["mean", "size"]).rename(columns={"mean": "keep_rate", "size": "n"})
rates_mode["keep_rate"] *= 100
print("Keep rate (%) by class and axis:")
display(rates_axis_class.round(1))
print("Keep rate (%) by axis:", rates_overall.round(1).to_dict())
print("Keep rate (%) by axis and mode:")
display(rates_mode.round(1))
rates_axis_class.round(2).to_csv(TABLE_DIR / "gen_keep_rates_by_class_axis.csv")

fig, ax = plt.subplots(figsize=(6, 3.2))
rates_overall.plot.bar(ax=ax, color="#4c78a8")
ax.set_ylabel("kept (%)"); ax.set_ylim(0, 100); ax.set_title("Generated images kept by the quality filter")
plt.xticks(rotation=0); plt.tight_layout(); save_fig(fig, "05_filter_keep_rates.png"); plt.show()

In [ ]:
# ---------------- Examples: kept vs rejected ----------------
def show_examples(frame, title, n=8, fname=None):
    if len(frame) == 0:
        print(f"({title}: none)")
        return
    s = frame.sample(min(n, len(frame)), random_state=cfg.SEED)
    fig, axes = plt.subplots(1, len(s), figsize=(2.5 * len(s), 3.1), squeeze=False)
    for ax, r in zip(axes[0], s.itertuples()):
        ax.imshow(Image.open(r.out_path)); ax.axis("off")
        ax.set_title(f"{r.label} | {r.axis}\nbase p={r.base_p_label:.2f} clip p={r.clip_p_label:.2f}\nclip says: {CLASS_NAMES[r.clip_top1]}", fontsize=7)
    fig.suptitle(title); plt.tight_layout()
    if fname:
        save_fig(fig, fname)
    plt.show()

show_examples(gm[gm.keep], "KEPT generated images", fname="06_filter_kept.png")
show_examples(gm[~gm.keep], "REJECTED generated images", fname="07_filter_rejected.png")

# trash and glass are NOT checked by CLIP (it is unreliable on them), so look at these with extra care
tg = gm[gm.label.isin(["trash", "glass"])]
show_examples(tg[tg.keep], "KEPT trash + glass (CLIP not used for these classes)", fname="07b_filter_kept_trash_glass.png")
show_examples(tg[~tg.keep], "REJECTED trash + glass (baseline sanity check only)", fname="07c_filter_rejected_trash_glass.png")

## MANUAL STEP 6: review the filter and (optionally) tune it
Look at the two rows of examples above and the keep-rate table.
- **Rejected images that actually look fine** (right object, interesting condition) mean the filter is too strict. Lower `BASE_MIN_PROB` / `CLIP_MIN_PROB` in CONFIG, or switch one judge off with `USE_BASE_FILTER` / `USE_CLIP_FILTER`.
- **Kept images that look wrong** mean it is too loose. Raise the thresholds.
- If one **axis or class has a very low keep rate**, that axis's prompts or strengths are probably too aggressive (go back to 5b / lower that axis's strengths).
- **Look at the trash and glass previews specifically** (the extra two rows of examples): only the baseline sanity check protects those classes, so wrong-looking kept images are most likely there. If they are bad, raise `BASE_MIN_PROB` or drop that class's generated images by hand.
- If *nothing* is kept, Section 7 will skip the generative models and tell you so.

After changing thresholds, re-run the two cells under "Score every generated image" (the images do not need regenerating). Change thresholds based on your eyes and the **validation** set only, not on test numbers.

# 7. Retrain and compare 4 models

| Tag | Model | Training data |
|---|---|---|
| **A_baseline** | (a) baseline | original TrashNet train only (already trained in Section 3) |
| **B_classical** | (b) classical only | originals + classical-augmented copies |
| **C_generative** | (c) generative only | originals + **kept** Stable Diffusion images |
| **D_classical_generative** | (d) classical + generative | originals + classical copies + kept Stable Diffusion images |

**What makes the comparison fair.** All four models use the same architecture, pretrained weights, seed, epochs, learning rate, batch size, flip-only base augmentation, and **the same clean validation set** for picking the best epoch. They differ *only* in the training images.

**Budget matching.** With `MATCH_BUDGET=True` the classical pool is trimmed so it has the same number of images per class and axis as the *kept* generative pool. So B and C add the same amount of extra data, and D adds both (twice as much extra data as B or C). The table reports `n_train` for every model so you can see this.

**Reminder.** Only train images were augmented. Val, test and TACO were never touched.

In [ ]:
import hashlib

def match_classical_budget(classical, kept):
    """Trim the classical pool to the same count per (class, axis) as the kept generative pool."""
    parts = []
    for (label, axis), g in kept.groupby(["label", "axis"]):
        pool = classical[(classical.label == label) & (classical.axis == axis)]
        parts.append(pool.sample(min(len(g), len(pool)), random_state=cfg.SEED))
    return pd.concat(parts) if parts else classical.iloc[0:0]

cols = ["path", "label", "label_id"]
orig_part = train_df[cols].assign(origin="original")

gen_kept = pd.read_csv(TABLE_DIR / "gen_kept.csv") if (TABLE_DIR / "gen_kept.csv").exists() else pd.DataFrame(columns=cols)
if len(gen_kept) and cfg.MATCH_BUDGET:
    classical_used = match_classical_budget(classical_pool, gen_kept)
else:
    classical_used = classical_pool
    if cfg.MATCH_BUDGET:
        print("Note: no generative images were kept, so the classical pool is NOT trimmed.")
classical_part = classical_used[cols].assign(origin="classical")
generative_part = gen_kept[cols].assign(origin="generative") if len(gen_kept) else pd.DataFrame(columns=cols + ["origin"])

TRAIN_SETS = {
    "A_baseline": orig_part,
    "B_classical": pd.concat([orig_part, classical_part]),
    "C_generative": pd.concat([orig_part, generative_part]),
    "D_classical_generative": pd.concat([orig_part, classical_part, generative_part]),
}
TRAIN_SETS = {k: v.reset_index(drop=True) for k, v in TRAIN_SETS.items()}
comp = pd.DataFrame({k: v.origin.value_counts() for k, v in TRAIN_SETS.items()}).fillna(0).astype(int).reindex(["original", "classical", "generative"]).fillna(0).astype(int)
comp.loc["TOTAL"] = comp.sum()
print("Training-set composition (number of images):")
display(comp)

In [ ]:
# ---------------- Train B, C, D (A was trained in Section 3) ----------------
for tag, frame in TRAIN_SETS.items():
    if tag == "A_baseline":
        continue
    if tag in ("C_generative", "D_classical_generative") and len(generative_part) == 0:
        print(f"[{tag}] SKIPPED: no generated images survived the quality filter.")
        continue
    if tag == "B_classical" and len(classical_part) == 0:
        print(f"[{tag}] SKIPPED: the classical pool is empty.")
        continue
    MODELS[tag], HISTORY[tag] = train_model(tag, frame, val_df)
    N_TRAIN[tag] = len(frame)
N_TRAIN["A_baseline"] = len(train_df)
TAGS = [t for t in TRAIN_SETS if t in MODELS or (MODEL_DIR / f"{t}_best.pt").exists()]
for t in TAGS:
    get_model(t)
print("Models available for evaluation:", TAGS)

In [ ]:
# ---------------- Training curves ----------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for tag in TAGS:
    h = HISTORY.get(tag)
    if h is None:
        h = pd.read_csv(TABLE_DIR / f"history_{tag}.csv")
    axes[0].plot(h.epoch, h.train_loss, marker="o", label=tag)
    axes[1].plot(h.epoch, h.val_macro_f1, marker="o", label=tag)
axes[0].set_title("Training loss"); axes[1].set_title("Validation macro-F1 (clean TrashNet val)")
for a in axes:
    a.set_xlabel("epoch")
axes[1].legend(fontsize=7)
plt.tight_layout(); save_fig(fig, "08_training_curves.png"); plt.show()

# 8. Evaluation

For every model we report, on **two** test sets (TrashNet test and the TACO real-world test):
- **Accuracy** and **macro-F1**, each with a **95% bootstrap confidence interval** (we re-sample the test images 1000 times to see how much the number could wobble just by luck). Small differences between models that fall inside each other's intervals should **not** be treated as real improvements.
- A **per-class report** and **confusion matrices** (rows = true class, columns = predicted class, each row normalised to 1; the number in a cell is the count).

**Important about statistics.** Each model was trained **once** with one seed. Run-to-run randomness in training can change results by a point or two, so a single-seed difference is only suggestive. For a stronger claim, repeat the notebook with a few different `SEED` values (the result files are saved in a per-mode folder, so change `DRIVE_ROOT` between repeats or copy `runs/` aside).

In [ ]:
def bootstrap_ci(y, p, fn, n=None, seed=0):
    """95% bootstrap confidence interval of a metric."""
    n = n or cfg.BOOTSTRAP_N
    rng = np.random.default_rng(seed)
    vals = [fn(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return tuple(np.percentile(vals, [2.5, 97.5]))

acc_fn = lambda y, p: float((y == p).mean())
f1_fn = lambda y, p: float(f1_score(y, p, labels=list(range(N_CLASSES)), average="macro", zero_division=0))

EVAL_SETS = {"trashnet_test": (X_tn_test, y_tn_test), "taco_test": (X_taco, y_taco)}
eval_rows, per_class_rows, CMS = [], [], {}
for tag in tqdm(TAGS, desc="Evaluating models"):
    for set_name, (X, y) in EVAL_SETS.items():
        pred = predict_proba(get_model(tag), X).argmax(1)
        m = compute_metrics(y, pred)
        a_lo, a_hi = bootstrap_ci(y, pred, acc_fn)
        f_lo, f_hi = bootstrap_ci(y, pred, f1_fn)
        eval_rows.append(dict(model=tag, set=set_name, n=len(y), acc=m["acc"], acc_lo=a_lo, acc_hi=a_hi, macro_f1=m["macro_f1"], f1_lo=f_lo, f1_hi=f_hi))
        CMS[(tag, set_name)] = m["cm"]
        for cname in CLASS_NAMES:
            r = m["report"][cname]
            per_class_rows.append(dict(model=tag, set=set_name, cls=cname, precision=r["precision"], recall=r["recall"], f1=r["f1-score"], support=int(r["support"])))

eval_df = pd.DataFrame(eval_rows)
per_class_df = pd.DataFrame(per_class_rows)
eval_df.to_csv(TABLE_DIR / "eval_main.csv", index=False)
per_class_df.to_csv(TABLE_DIR / "per_class_report.csv", index=False)
print("Accuracy / macro-F1 with 95% bootstrap CI:")
display(eval_df.round(3))

for set_name in EVAL_SETS:
    print(f"\nPer-class F1 on {set_name}:")
    display(per_class_df[per_class_df.set == set_name].pivot(index="cls", columns="model", values="f1").reindex(CLASS_NAMES)[TAGS].round(3))
    print("Support (number of test images per class):", per_class_df[(per_class_df.set == set_name) & (per_class_df.model == TAGS[0])].set_index("cls").support.to_dict())

In [ ]:
# ---------------- Confusion matrices ----------------
fig, axes = plt.subplots(2, len(TAGS), figsize=(3.6 * len(TAGS), 7.2), squeeze=False)
for r, set_name in enumerate(EVAL_SETS):
    for c, tag in enumerate(TAGS):
        cm = CMS[(tag, set_name)]
        norm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
        ax = axes[r, c]
        ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
        ax.set_xticklabels(CLASS_NAMES, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
        for i in range(N_CLASSES):
            for j in range(N_CLASSES):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6, color="white" if norm[i, j] > 0.5 else "black")
        ax.set_title(f"{tag}\n{set_name}", fontsize=8)
        if c == 0:
            ax.set_ylabel("true class")
        if r == 1:
            ax.set_xlabel("predicted class")
plt.tight_layout(); save_fig(fig, "09_confusion_matrices.png"); plt.show()

### Robustness test: controlled corruptions of the clean TrashNet test set

TACO is "real" but we do not know *which* kind of difficulty hurts. So we also take the clean TrashNet **test** images and corrupt them in a controlled way, one axis at a time, at three severities (1 = mild, 3 = strong):

| Axis | Corruptions |
|---|---|
| Lighting | dim, colour cast (warm or cold), glare spot, shadow band |
| Contamination | noise, blur, occlusion patches, stain overlay |
| Material condition | rotation, perspective, crumple-like warp, partial crop |

Each corruption is deterministic (seeded per image), so every model sees **exactly the same corrupted images**.

**Read this honestly.** These corruptions belong to the *same families* of effects as the classical augmentations in Section 4 (but with fixed severities and different random draws). So the classical-augmentation models have a **home-field advantage** here. The generative images contain these conditions in a different, more realistic form, so they are not favoured in the same way. A model can look good on this test simply because it saw similar synthetic effects. TACO is the more independent check, because it was captured in real conditions.

In [ ]:
# ---------------- Controlled corruptions (severity 1..3) ----------------
def _sev(vals, s):
    return vals[s - 1]

def c_dim(img, s, rng):
    return clip_u8(img.astype(np.float32) * _sev([0.65, 0.45, 0.28], s))

def c_color_cast(img, s, rng):
    k = _sev([0.12, 0.22, 0.35], s) * (1 if rng.random() < 0.5 else -1)     # warm or cold
    out = img.astype(np.float32); out[..., 0] *= 1 + k; out[..., 2] *= 1 - k
    return clip_u8(out)

def c_glare(img, s, rng):
    h, w = img.shape[:2]
    cx, cy = rng.uniform(0.25, 0.75) * w, rng.uniform(0.25, 0.75) * h
    r = _sev([0.15, 0.22, 0.30], s) * w
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    spot = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (r ** 2)))
    return clip_u8(img.astype(np.float32) + 255.0 * _sev([0.5, 0.75, 1.0], s) * spot[..., None])

def c_shadow(img, s, rng):
    h, w = img.shape[:2]
    ang = rng.uniform(0, np.pi)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    d = (xx - w / 2) * np.cos(ang) + (yy - h / 2) * np.sin(ang)
    mask = 1 / (1 + np.exp(-d / (0.04 * w)))                             # soft edge half-plane
    return clip_u8(img.astype(np.float32) * (1 - _sev([0.25, 0.4, 0.55], s) * mask[..., None]))

def c_noise(img, s, rng):
    return clip_u8(img.astype(np.float32) + rng.normal(0, _sev([10, 25, 45], s), img.shape))

def c_blur(img, s, rng):
    return cv2.GaussianBlur(img, (0, 0), _sev([1.0, 2.0, 3.5], s))

def c_occlusion(img, s, rng):
    h, w = img.shape[:2]
    out = img.copy()
    for _ in range(_sev([1, 2, 3], s)):
        side = int(_sev([0.15, 0.20, 0.25], s) * min(h, w))
        x0, y0 = int(rng.uniform(0, w - side)), int(rng.uniform(0, h - side))
        out[y0:y0 + side, x0:x0 + side] = rng.integers(30, 180, 3)
    return out

def c_stain(img, s, rng):
    h, w = img.shape[:2]
    f = smooth_field(h, w, 48, rng)
    mask = cv2.GaussianBlur(np.clip((f - _sev([0.78, 0.70, 0.62], s)) / 0.10, 0, 1), (0, 0), 2.0)[..., None]
    col = np.array(STAIN_COLORS[int(rng.integers(len(STAIN_COLORS)))], np.float32)
    a = _sev([0.35, 0.55, 0.75], s)
    return clip_u8(img.astype(np.float32) * (1 - a * mask) + col * a * mask)

def c_rotate(img, s, rng):
    h, w = img.shape[:2]
    deg = _sev([15, 30, 45], s) * (1 if rng.random() < 0.5 else -1)
    return cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), deg, 1.0), (w, h), borderMode=cv2.BORDER_REFLECT_101)

def c_perspective(img, s, rng):
    h, w = img.shape[:2]
    m = _sev([0.06, 0.12, 0.20], s) * min(h, w)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = src + (rng.choice([-1, 1], src.shape) * m).astype(np.float32)
    return cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h), borderMode=cv2.BORDER_REFLECT_101)

def c_crumple(img, s, rng):
    return np_elastic(img, rng, amp=_sev([0.025, 0.05, 0.08], s) * min(img.shape[:2]))

def c_partial_crop(img, s, rng):
    h, w = img.shape[:2]
    frac = _sev([0.85, 0.70, 0.55], s)
    ch, cw = int(h * frac), int(w * frac)
    y0, x0 = int(rng.uniform(0, h - ch)), int(rng.uniform(0, w - cw))
    return cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_LINEAR)

CORRUPTIONS = {
    "lighting": {"dim": c_dim, "color_cast": c_color_cast, "glare": c_glare, "shadow": c_shadow},
    "contamination": {"noise": c_noise, "blur": c_blur, "occlusion": c_occlusion, "stain": c_stain},
    "condition": {"rotate": c_rotate, "perspective": c_perspective, "crumple": c_crumple, "partial_crop": c_partial_crop},
}

def corrupt_array(X, fn, kind, sev):
    """Apply one corruption to every image; the random draws depend only on (kind, severity, image index)."""
    return np.stack([fn(X[i], sev, np.random.default_rng(stable_seed(cfg.SEED, "robust", kind, sev, i))) for i in range(len(X))])

# visual check: one test image under every corruption at severity 3
fig, axes = plt.subplots(3, 5, figsize=(11, 7))
base_img = X_tn_test[0]
for r, (axis, kinds) in enumerate(CORRUPTIONS.items()):
    axes[r, 0].imshow(base_img); axes[r, 0].set_title("clean", fontsize=8)
    for c, (kind, fn) in enumerate(kinds.items()):
        axes[r, c + 1].imshow(corrupt_array(base_img[None], fn, kind, 3)[0]); axes[r, c + 1].set_title(f"{axis}: {kind} (sev 3)", fontsize=7)
for a in axes.ravel():
    a.axis("off")
plt.tight_layout(); save_fig(fig, "10_robustness_corruptions.png"); plt.show()

In [ ]:
# ---------------- Run the robustness test for every model ----------------
robust_rows = []
for tag in TAGS:     # clean reference (severity 0)
    acc0 = acc_fn(y_tn_test, predict_proba(get_model(tag), X_tn_test).argmax(1))
    for axis in AXES:
        robust_rows.append(dict(model=tag, axis=axis, kind="clean", severity=0, acc=acc0))

jobs = [(axis, kind, fn, sev) for axis, kinds in CORRUPTIONS.items() for kind, fn in kinds.items() for sev in cfg.ROBUST_SEVERITIES]
for axis, kind, fn, sev in tqdm(jobs, desc="Robustness test"):
    Xc = corrupt_array(X_tn_test, fn, kind, sev)               # build once, evaluate all models on identical images
    for tag in TAGS:
        acc = acc_fn(y_tn_test, predict_proba(get_model(tag), Xc).argmax(1))
        robust_rows.append(dict(model=tag, axis=axis, kind=kind, severity=sev, acc=acc))

robust_df = pd.DataFrame(robust_rows)
robust_df.to_csv(TABLE_DIR / "robustness_long.csv", index=False)
robust_axis = robust_df[robust_df.severity > 0].groupby(["model", "axis"]).acc.mean().unstack()[AXES].reindex(TAGS)
robust_axis["robust_mean"] = robust_axis.mean(axis=1)
print("Mean accuracy under corruption (average over corruption types and severities):")
display(robust_axis.round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, axis in zip(axes, AXES):
    for tag in TAGS:
        d = robust_df[(robust_df.model == tag) & (robust_df.axis == axis)].groupby("severity").acc.mean()
        ax.plot(d.index, d.values, marker="o", label=tag)
    ax.set_title(f"{axis}"); ax.set_xlabel("severity (0 = clean)"); ax.set_xticks([0] + list(cfg.ROBUST_SEVERITIES))
axes[0].set_ylabel("accuracy"); axes[0].legend(fontsize=7)
plt.tight_layout(); save_fig(fig, "11_robustness_by_severity.png"); plt.show()

In [ ]:
# ---------------- Final comparison table and bar chart ----------------
def get_row(tag, set_name):
    return eval_df[(eval_df.model == tag) & (eval_df.set == set_name)].iloc[0]

final = pd.DataFrame(index=TAGS)
final["n_train"] = [N_TRAIN.get(t, np.nan) for t in TAGS]
final["trashnet_acc"] = [get_row(t, "trashnet_test").acc for t in TAGS]
final["trashnet_macro_f1"] = [get_row(t, "trashnet_test").macro_f1 for t in TAGS]
final["taco_acc"] = [get_row(t, "taco_test").acc for t in TAGS]
final["taco_macro_f1"] = [get_row(t, "taco_test").macro_f1 for t in TAGS]
for ax_ in AXES:
    final[f"robust_{ax_}"] = robust_axis[ax_]
final["robust_mean"] = robust_axis["robust_mean"]
final["trashnet_acc_ci"] = [f"[{get_row(t, 'trashnet_test').acc_lo:.3f}, {get_row(t, 'trashnet_test').acc_hi:.3f}]" for t in TAGS]
final["taco_acc_ci"] = [f"[{get_row(t, 'taco_test').acc_lo:.3f}, {get_row(t, 'taco_test').acc_hi:.3f}]" for t in TAGS]
final.index.name = "model"
final.round(4).to_csv(TABLE_DIR / "results.csv")
print("FINAL COMPARISON (all numbers computed above):")
display(final.round(3))

metrics_to_plot = [("trashnet_acc", "TrashNet\naccuracy"), ("taco_acc", "TACO\naccuracy"), ("taco_macro_f1", "TACO\nmacro-F1"),
                   ("robust_lighting", "Robust:\nlighting"), ("robust_contamination", "Robust:\ncontamination"), ("robust_condition", "Robust:\ncondition")]
palette = ["#4c78a8", "#f58518", "#54a24b", "#b279a2"]
fig, ax = plt.subplots(figsize=(12, 4.2))
width = 0.8 / len(TAGS)
for k, tag in enumerate(TAGS):
    vals = [final.loc[tag, m] for m, _ in metrics_to_plot]
    bars = ax.bar(np.arange(len(vals)) + k * width, vals, width, label=tag, color=palette[k % 4])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=6)
ax.set_xticks(np.arange(len(metrics_to_plot)) + width * (len(TAGS) - 1) / 2)
ax.set_xticklabels([n for _, n in metrics_to_plot]); ax.set_ylim(0, 1.08); ax.set_ylabel("score")
ax.set_title("Baseline vs augmented models"); ax.legend(fontsize=8, ncol=len(TAGS)); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout(); save_fig(fig, "12_final_comparison.png"); plt.show()

In [ ]:
# ---------------- Pick the "best" model (for the demo and export) ----------------
BEST_COL = {"robust_mean": "robust_mean", "taco_f1": "taco_macro_f1", "trashnet_f1": "trashnet_macro_f1"}[cfg.BEST_BY]
best_tag = final[BEST_COL].astype(float).idxmax()
print(f"Best model by '{cfg.BEST_BY}': {best_tag}  ({BEST_COL} = {final.loc[best_tag, BEST_COL]:.3f})")
print("Note: this only decides which model is exported / used in the demo. The table above still reports unbiased numbers for ALL models.")
print("If the baseline wins, that is a valid result: it means augmentation did not help under this setup (see the caveats in Section 10).")

### Optional: Grad-CAM (where is the model looking?)
Grad-CAM draws a heat-map over the image showing which regions pushed the model toward its answer. Red = strong influence. It is a *visual explanation*, not proof: use it to spot cases like "the model looks at the conveyor belt instead of the item". We show real TACO test images for the baseline and the best model. Turn it off with `RUN_GRADCAM=False`.

In [ ]:
import torch.nn.functional as F

def grad_cam(model, img_u8):
    """Return (heatmap in [0,1] with the image's height/width, predicted class index)."""
    model.eval()
    layer = model.layer4[-1] if hasattr(model, "layer4") else model.features[-1]
    store = {}
    def fwd_hook(module, inp, out):
        store["a"] = out
        out.register_hook(lambda g: store.__setitem__("g", g))
    handle = layer.register_forward_hook(fwd_hook)
    try:
        logits = model(to_tensor_batch(img_u8[None]))
        cls = int(logits.argmax(1))
        model.zero_grad(set_to_none=True)
        logits[0, cls].backward()
    finally:
        handle.remove()
    weights = store["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=img_u8.shape[:2], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
    return cam.detach().cpu().numpy(), cls

def overlay(img_u8, cam):
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return clip_u8(0.55 * img_u8 + 0.45 * heat)

if cfg.RUN_GRADCAM:
    n_show = min(6, len(X_taco))
    pick = np.linspace(0, len(X_taco) - 1, n_show).astype(int)          # evenly spread, not cherry-picked
    show_tags = list(dict.fromkeys(["A_baseline", best_tag]))
    fig, axes = plt.subplots(1 + len(show_tags), n_show, figsize=(2.4 * n_show, 2.5 * (1 + len(show_tags))), squeeze=False)
    for j, i in enumerate(pick):
        axes[0, j].imshow(X_taco[i]); axes[0, j].set_title(f"true: {CLASS_NAMES[y_taco[i]]}", fontsize=8)
        for r, tag in enumerate(show_tags):
            cam, cls = grad_cam(get_model(tag), X_taco[i])
            axes[r + 1, j].imshow(overlay(X_taco[i], cam))
            axes[r + 1, j].set_title(f"{tag}\npred: {CLASS_NAMES[cls]}", fontsize=7, color="green" if cls == y_taco[i] else "crimson")
    for a in axes.ravel():
        a.axis("off")
    plt.tight_layout(); save_fig(fig, "13_gradcam_taco.png"); plt.show()

# 9. Gradio demo

**What it does.** A small web app inside Colab: you upload a photo of a waste item, choose a condition (**lighting**, **contamination** or **material condition**), and it shows
- the original plus several augmented versions (classical augmentation, instantly), optionally also **Stable Diffusion** versions (slow, needs the GPU and loads the model again), and
- the prediction of the **best model** for the original and for every augmented version, so you can see how fragile the classifier is.

**MANUAL STEP 9:** run the last cell of this section, click the public `gradio.live` link it prints (or use the inline app), and try a few photos, including your own phone photos of real waste. Take screenshots for your report. The link works only while the cell is running.

In [ ]:
import gradio as gr

AXIS_CHOICES = {"Lighting": "lighting", "Contamination": "contamination", "Material condition": "condition"}
MATERIAL_CHOICES = ["auto (use the classifier's guess)"] + CLASS_NAMES
DEMO_TMP = ROOT / "demo_tmp"
DEMO_TMP.mkdir(exist_ok=True)
DEMO_MODEL = get_model(best_tag)
print("Demo classifier:", best_tag)

def predict_np(img_u8):
    """Class probabilities for any RGB uint8 image."""
    x = np.asarray(Image.fromarray(img_u8).convert("RGB").resize((cfg.IMG_SIZE, cfg.IMG_SIZE), Image.BILINEAR))
    return predict_proba(DEMO_MODEL, x[None])[0]

def generate_variant(img_u8, material, axis, seed):
    """One Stable Diffusion image (loads the pipeline the first time)."""
    if not GEN:
        load_generators()
    rng = np.random.default_rng(seed)
    use_inpaint = len(INPAINT_BANK[material][axis]) > 0 and rng.random() < cfg.GEN_INPAINT_FRACTION.get(axis, 0.0)
    bank = INPAINT_BANK[material][axis] if use_inpaint else PROMPT_BANK[material][axis]
    strengths = cfg.GEN_INPAINT_STRENGTHS if use_inpaint else cfg.GEN_STRENGTHS[axis]
    tmp_path = DEMO_TMP / f"upload_{seed}.jpg"
    Image.fromarray(img_u8).convert("RGB").save(tmp_path, quality=95)
    task = dict(src_path=str(tmp_path), mode="inpaint" if use_inpaint else "img2img", prompt=bank[int(rng.integers(len(bank)))],
                strength=float(strengths[int(rng.integers(len(strengths)))]), guidance=float(cfg.GEN_GUIDANCE[int(rng.integers(len(cfg.GEN_GUIDANCE)))]), seed=int(seed))
    return np.asarray(safe_generate([task])[0].convert("RGB"))

def demo_fn(image, axis_label, n_variants, use_gen, material):
    if image is None:
        return [], {}, "Please upload an image first."
    axis = AXIS_CHOICES[axis_label]
    img = np.asarray(Image.fromarray(image).convert("RGB"))
    p0 = predict_np(img)
    gallery = [(img, f"original: {CLASS_NAMES[int(p0.argmax())]} ({p0.max():.2f})")]
    lines = [f"**Original** -> {CLASS_NAMES[int(p0.argmax())]} ({p0.max():.2f})"]
    base_seed = time.time_ns() % (2**31 - 100)
    for k in range(int(n_variants)):                                    # classical variants
        aug, ops = apply_classical(img, axis, np.random.default_rng(base_seed + k))
        p = predict_np(aug)
        gallery.append((aug, f"classical {'+'.join(ops)}: {CLASS_NAMES[int(p.argmax())]} ({p.max():.2f})"))
        lines.append(f"- classical `{'+'.join(ops)}` -> {CLASS_NAMES[int(p.argmax())]} ({p.max():.2f})")
    if use_gen:                                                         # Stable Diffusion variants (slow)
        mat = material if material in CLASS_NAMES else CLASS_NAMES[int(p0.argmax())]
        try:
            for k in range(min(int(n_variants), 3)):
                g = generate_variant(img, mat, axis, base_seed + 1000 + k)
                p = predict_np(g)
                gallery.append((g, f"Stable Diffusion: {CLASS_NAMES[int(p.argmax())]} ({p.max():.2f})"))
                lines.append(f"- Stable Diffusion variant {k + 1} -> {CLASS_NAMES[int(p.argmax())]} ({p.max():.2f})")
        except Exception as e:
            lines.append(f"- Stable Diffusion failed: `{type(e).__name__}: {str(e)[:120]}` (needs a GPU and enough memory)")
    return gallery, {CLASS_NAMES[i]: float(p0[i]) for i in range(N_CLASSES)}, "\n".join(lines)

# smoke test without the web UI (classical only)
_g, _lab, _txt = demo_fn(X_tn_test[0], "Lighting", 2, False, MATERIAL_CHOICES[0])
print(_txt)

In [ ]:
with gr.Blocks(title="SortAugment AI") as demo:
    gr.Markdown("# SortAugment AI: Recyclable Material Image Augmenter\nUpload a photo of a waste item, pick a condition, and see how the classifier reacts to augmented versions.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="numpy", label="Upload a photo of a waste item")
            axis_dd = gr.Dropdown(choices=list(AXIS_CHOICES), value="Lighting", label="Condition to simulate")
            n_sl = gr.Slider(minimum=1, maximum=6, value=4, step=1, label="Number of augmented versions")
            use_gen = gr.Checkbox(value=False, label="Also use Stable Diffusion (slow; needs the GPU and loads the model)")
            mat_dd = gr.Dropdown(choices=MATERIAL_CHOICES, value=MATERIAL_CHOICES[0], label="Material (only used for Stable Diffusion prompts)")
            btn = gr.Button("Augment and classify")
        with gr.Column():
            gal = gr.Gallery(label="Original and augmented versions (with predictions)", columns=3)
            lab = gr.Label(num_top_classes=N_CLASSES, label="Prediction for the original image")
            txt = gr.Markdown()
    btn.click(demo_fn, inputs=[inp, axis_dd, n_sl, use_gen, mat_dd], outputs=[gal, lab, txt])

demo.launch(share=True, debug=False)     # prints a public link; in Colab the cell keeps the app alive

In [ ]:
# Run this when you are done with the demo (frees the port / public link).
demo.close()

# 10. Export: results, figures, best model, requirements, summary

This section collects everything a report or a viva needs into one folder (and one zip) in Drive:
- `results.csv`, plus the other result tables
- all figures (PNG)
- `best_model.pt` (weights + class names + image size; a loading snippet is in `summary.md`)
- `requirements.txt` (the exact versions installed in this Colab session)
- `config.json` (every setting of the run)
- `summary.md`: an **auto-generated** summary. Every number in it is read from the result tables, and the wording ("higher", "lower", "no clear difference") is chosen by the code from those numbers. Nothing is written by hand, so re-running with new settings gives a new honest summary.

In [ ]:
import importlib.metadata as md

# ---- 1) tables, figures, best model ----
TABLES_TO_COPY = ["results.csv", "eval_main.csv", "per_class_report.csv", "robustness_long.csv", "baseline_before.csv",
                  "gen_metadata.csv", "gen_quality_scores.csv", "gen_keep_rates_by_class_axis.csv", "classical_pool.csv",
                  "trashnet_splits.csv", "taco_test.csv"]
(EXPORT_DIR / "tables").mkdir(exist_ok=True)
for name in TABLES_TO_COPY:
    if (TABLE_DIR / name).exists():
        shutil.copy2(TABLE_DIR / name, EXPORT_DIR / "tables" / name)
shutil.copy2(TABLE_DIR / "results.csv", EXPORT_DIR / "results.csv")
shutil.copytree(FIG_DIR, EXPORT_DIR / "figures", dirs_exist_ok=True)
shutil.copy2(MODEL_DIR / f"{best_tag}_best.pt", EXPORT_DIR / "best_model.pt")
if (GEN_DIR / "prompt_bank.json").exists():
    shutil.copy2(GEN_DIR / "prompt_bank.json", EXPORT_DIR / "prompt_bank.json")

# ---- 2) requirements.txt (versions actually installed) ----
PKGS = ["torch", "torchvision", "diffusers", "transformers", "accelerate", "safetensors", "albumentations", "opencv-python-headless",
        "opencv-python", "scikit-learn", "matplotlib", "pandas", "numpy", "Pillow", "gradio", "datasets", "huggingface_hub", "tqdm", "requests"]
req_lines = []
for p in PKGS:
    try:
        req_lines.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError:
        pass
(EXPORT_DIR / "requirements.txt").write_text("\n".join(req_lines) + "\n")

# ---- 3) config.json ----
with open(EXPORT_DIR / "config.json", "w") as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in CONFIG.items()}, f, indent=2, default=str)
print("requirements.txt:\n" + "\n".join(req_lines))

In [ ]:
# ---- 4) auto-generated summary.md ----
def md_table(df, floatfmt=3):
    cols = ["model"] + list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx, row in df.iterrows():
        cells = [str(idx)] + [f"{v:.{floatfmt}f}" if isinstance(v, (float, np.floating)) else str(v) for v in row]
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)

def describe_delta(delta_pp, tol=0.5):
    if delta_pp > tol:
        return f"higher by {delta_pp:.1f} percentage points"
    if delta_pp < -tol:
        return f"lower by {abs(delta_pp):.1f} percentage points"
    return "about the same (within 0.5 percentage points)"

def ci_overlap(tag_a, tag_b, set_name, metric="acc"):
    a, b = get_row(tag_a, set_name), get_row(tag_b, set_name)
    lo, hi = ("acc_lo", "acc_hi") if metric == "acc" else ("f1_lo", "f1_hi")
    return not (a[lo] > b[hi] or b[lo] > a[hi])

def make_summary():
    L = []
    L.append("# SortAugment AI: results summary (auto-generated)\n")
    L.append(f"- Generated on: {time.strftime('%Y-%m-%d %H:%M')} | run mode: **{cfg.RUN_NAME.upper()}** | seed: {cfg.SEED} | classifier: {cfg.MODEL_NAME} | epochs: {cfg.EPOCHS_EFF}")
    if cfg.DEBUG:
        L.append("\n> **DEBUG RUN: the datasets were cut to a handful of images, so these numbers say nothing about real performance. Set DEBUG=False and rerun for the real experiment.**\n")
    L.append(f"- TrashNet split (train/val/test): {len(train_df)}/{len(val_df)}/{len(test_df)} images. TACO test crops: {len(taco_test)}.")
    gm_ = pd.read_csv(TABLE_DIR / "gen_quality_scores.csv") if (TABLE_DIR / "gen_quality_scores.csv").exists() else None
    if gm_ is not None and len(gm_):
        models_used = ", ".join(f"{k} ({v})" for k, v in gm_.model.value_counts().items())
        L.append(f"- Generative images: {len(gm_)} made with {models_used}; {int(gm_.keep.sum())} kept by the quality filter ({100 * gm_.keep.mean():.1f}%).")
        per_axis = gm_.groupby("axis").keep.mean().mul(100).round(1).to_dict()
        L.append(f"- Keep rate by axis (%): {per_axis}.")
    L.append("\n## Results table\n")
    L.append(md_table(final.drop(columns=["trashnet_acc_ci", "taco_acc_ci"]).round(3)))
    L.append("\n95% bootstrap confidence intervals for accuracy:\n")
    L.append(md_table(final[["trashnet_acc_ci", "taco_acc_ci"]]))
    L.append("\n## Comparison with the baseline (A_baseline)\n")
    for tag in TAGS:
        if tag == "A_baseline":
            continue
        L.append(f"**{tag}** (n_train = {int(final.loc[tag, 'n_train'])} vs {int(final.loc['A_baseline', 'n_train'])} for the baseline):")
        for col, label, set_name, metric in [("trashnet_acc", "TrashNet test accuracy", "trashnet_test", "acc"), ("taco_acc", "TACO test accuracy", "taco_test", "acc"),
                                             ("taco_macro_f1", "TACO macro-F1", "taco_test", "f1"), ("robust_lighting", "robustness to lighting", None, None),
                                             ("robust_contamination", "robustness to contamination", None, None), ("robust_condition", "robustness to material condition", None, None)]:
            d = 100 * (final.loc[tag, col] - final.loc["A_baseline", col])
            note = ""
            if set_name is not None and abs(d) > 0.5 and ci_overlap(tag, "A_baseline", set_name, metric):
                note = " (95% confidence intervals overlap, so this is not a clear difference)"
            L.append(f"- {label}: {describe_delta(d)}{note}.")
        L.append("")
    L.append(f"## Best model\n\nBy `{cfg.BEST_BY}` the best model is **{best_tag}** ({BEST_COL} = {final.loc[best_tag, BEST_COL]:.3f}). It is saved as `best_model.pt`.\n")
    L.append("## Caveats (read before quoting any number)\n")
    L += ["- Each model was trained once with one seed; run-to-run variation can be a point or two. Repeat with other seeds for a stronger claim.",
          "- The classical pool is trimmed to match the kept generative pool (MATCH_BUDGET); model D therefore gets about twice as much extra data as B or C.",
          "- The robustness test uses the same kinds of distortion as the classical augmentation (dimming, glare, shadows, noise, blur, occlusion, stains), so the classical models have a home-field advantage there. The fairer evidence is the TACO result and real photos of your own (try them in the Section 9 demo).",
          "- TACO labels come from our own category mapping (`TACO_TO_6` in the notebook). Items with no matching class or an arguable label (battery, food waste, shoe, rope, cigarette, unlabeled litter, drink carton, tissues, plastified paper bag, aluminium blister pack) were excluded on purpose (`TACO_EXCLUDED`); other reasonable mappings would change the numbers.",
          "- The quality filter uses CLIP and (with a low threshold, as a sanity check) the baseline classifier; both can wrongly reject valid hard images (e.g. very dim ones). Trash and glass are exempt from the CLIP judge because CLIP was unreliable on them, so those two classes rely on the baseline sanity check alone." + (" In DEBUG the baseline judge was switched off." if cfg.DEBUG else ""),
          "- Diffusion-generated images can still carry subtle label noise that the filter does not catch."]
    L.append("\n## Loading the exported model\n\n```python\nimport torch\nfrom torchvision import models\nck = torch.load('best_model.pt', map_location='cpu', weights_only=False)\n"
             "m = models.resnet18(); m.fc = torch.nn.Linear(m.fc.in_features, len(ck['class_names']))   # use mobilenet_v2 if ck['arch'] says so\n"
             "m.load_state_dict(ck['state_dict']); m.eval()   # input: RGB, ck['img_size'] square, ImageNet mean/std normalisation\n```\n")
    return "\n".join(L)

summary_text = make_summary()
(EXPORT_DIR / "summary.md").write_text(summary_text)
print(summary_text)

In [ ]:
# ---- 5) zip everything ----
zip_path = shutil.make_archive(str(ROOT / f"SortAugment_export_{cfg.RUN_NAME}"), "zip", EXPORT_DIR)
print("Export folder:", EXPORT_DIR)
print("Zip file     :", zip_path)
print("\nContents:")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(EXPORT_DIR)}  ({p.stat().st_size / 1e3:.0f} KB)")

DOWNLOAD_ZIP = False      # set True to also download the zip to your computer from Colab
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## Final checklist (MANUAL STEPS you still own)
- [ ] Set `DEBUG = False`, re-run Sections 2 to 10 for the real experiment (re-running is safe: finished work is skipped).
- [ ] Review the TACO class mapping (2b) and the prompt bank (5b); look at generated images (5c) and filter examples (6).
- [ ] Open `summary.md` and `results.csv`; write your report from *your* numbers and mention the caveats.
- [ ] Add the faculty rubric items (MANUAL STEP 0) and check each one is covered.
- [ ] Download the export zip from Drive (or set `DOWNLOAD_ZIP = True`).
- [ ] Cite the datasets: TrashNet (G. Thung and M. Yang, github.com/garythung/trashnet) and TACO (Proenca and Simoes, arXiv:2003.06975).